# PAD reproduction on Kaggle: Prompt-Anchored Vision–Text Distillation for Lifelong Person Re-ID (CVPR 2026)

This notebook **re-implements the official PAD code (github.com/zu-zi/PAD) inside one file**, so nothing has to be
cloned or downloaded from GitHub. The training logic, hyper-parameters, losses, prompt pool, and evaluation
protocol follow the official `configs/pad.yml`, `train_lifelong.py`, `test_lifelong.py`, and `pad/*` modules.
Only the infrastructure is new: Kaggle data discovery and conversion, checkpoint/resume across sessions, a
memory-safe evaluator that gives identical numbers, and result tables and plots.

**What it reproduces**: pick one with `PROTOCOL` in the config cell.
* **`"kaggle-4"` (default):** uses only datasets that are on Kaggle, i.e. **Market-1501 → CUHK-SYSU →
  DukeMTMC-reID → CUHK03**. This is AKA order-1 with MSMT17 left out (no Kaggle copy of the MSMT17 images exists).
  Every hyper-parameter, loss, and per-domain setting stays exactly as in the paper's code, so the only difference
  is one fewer domain in the sequence.
* **`"paper-5"`:** the full AKA order-1 sequence, Market-1501 → CUHK-SYSU → DukeMTMC-reID → MSMT17 → CUHK03. It
  needs MSMT17, which you have to request from its authors and upload yourself.
* After every step the notebook evaluates all domains seen so far (mAP / Rank-1). No stored exemplars are used.
* **Unseen domains (optional):** CUHK01, CUHK02, GRID, i-LIDS, PRID, SenseReID, and VIPeR. None of these are on
  Kaggle, so this part is skipped automatically unless you upload them.

**How to run on Kaggle**
1. *Settings → Accelerator → GPU T4 x2* (the notebook pins **one** GPU, as the authors did for every reported
   number) and *Internet → On* (needed for the CLIP ViT-B/16 weights).
2. *Add Input* → attach the datasets listed in the next cell. The notebook finds them anywhere under `/kaggle/input`.
3. Do a quick smoke test with `RUN_MODE = "quick"`, then set `RUN_MODE = "full"` and use **Save Version → Save & Run All
   (commit)** so it runs in the background (up to 12 h).
4. If a session ends early, attach this notebook's own output as an input and run again. Domains that already
   finished are found and skipped (see `RESUME`).

**Estimated full-run time on one T4:** `kaggle-4` takes about 6–8 h; `paper-5` takes about 9–11 h (MSMT17 alone
is about 2.5 h). The first pass over each dataset is slower because Kaggle reads `/kaggle/input` from network
storage. If the 12 h limit is hit, use the resume step above.

## Datasets to attach (Kaggle "Add Input")

**For the default `kaggle-4` protocol, attach these four Kaggle datasets** (set the notebook to `RUN_MODE = "full"` for real results):

| Step | Domain | Kaggle dataset (search the slug in *Add Input*) | Notes |
|---|---|---|---|
| 1 | Market-1501 | [`pengcw1/market-1501`](https://www.kaggle.com/datasets/pengcw1/market-1501) (alt: `rayiooo/reid_market-1501`) | standard `bounding_box_train / query / bounding_box_test` (confirmed on Kaggle) |
| 2 | CUHK-SYSU | [`manaschaiaonon/cuhk-sysu`](https://www.kaggle.com/datasets/manaschaiaonon/cuhk-sysu) | raw person-search release is converted to the AKA `cuhksysu4reid` splits automatically. If its layout is not recognised, the data cell prints its folder tree |
| 3 | DukeMTMC-reID | [`whurobin/dukemtmcreid`](https://www.kaggle.com/datasets/whurobin/dukemtmcreid) (alt: `igorkrashenyi/dukemtmc-reid`) | standard layout (confirmed on Kaggle) |
| 4 | CUHK03 | [`priyanagda/cuhk03`](https://www.kaggle.com/datasets/priyanagda/cuhk03) | raw `cuhk-03.mat` (confirmed on Kaggle), converted here with the CUHK03-NP (new-protocol, *detected*) split. The small split file `cuhk03_new_protocol_config_detected.mat` is used if attached, otherwise it is downloaded from its original source (Zhong et al., person-re-ranking), so keep Internet on |

**Do not attach `achrafabid7/msmt17`:** despite its name it contains a pretrained CLIP-ReID *model*, not images.

**Only needed for `paper-5` / extra unseen sets (not on Kaggle; upload as private datasets):**

| Domain | Source | Notes |
|---|---|---|
| MSMT17 | request form at http://www.pkuvmc.com/publications/msmt17.html | V1 (`train/ test/ list_*.txt`) or V2 (`mask_train_v2/ mask_test_v2/`); both handled |
| Unseen sets | DualNorm (BMVC'19) test-set bundle: Google Drive `1-5JqXxqQ14MCngGLMPqiA-zbI2CUBFxV` | VIPeR `cam_a/cam_b`, GRID `probe/gallery + features_and_partitions.mat`, PRID `single_shot`, i-LIDS `i-LIDS_Pedestrian/Persons`, CUHK01 `campus`, CUHK02 `Dataset/P5`, SenseReID `test_probe/test_gallery` |

If auto-discovery picks the wrong folder, set `DATA_OVERRIDES` in the config cell. These slugs were found by web
search, so check that the folder layout matches before you run.

In [ ]:
# ---- MUST be the first code cell: pin a single GPU (the authors report all numbers on one GPU; with >1 visible GPU
# ---- the official code wraps nn.DataParallel, which the README warns weakens distillation).
import os
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("PAD_GPU", "0")

import copy, csv, glob, json, logging, math, random, re, shutil, statistics, sys, time, urllib.request, hashlib
import os.path as osp
from bisect import bisect_right
from collections import OrderedDict, defaultdict
from datetime import timedelta
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.sampler import Sampler
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode

try:
    import timm
    from timm.data.random_erasing import RandomErasing
    from timm.scheduler.cosine_lr import CosineLRScheduler
except ImportError:  # timm is pre-installed on Kaggle; install it only if missing (PyPI, not GitHub)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm
    from timm.data.random_erasing import RandomErasing
    from timm.scheduler.cosine_lr import CosineLRScheduler

import warnings
# The official code steps the LR scheduler at the start of each epoch (kept for fidelity) -> silence PyTorch's notice.
warnings.filterwarnings("ignore", message="Detected call of `lr_scheduler.step()` before")
warnings.filterwarnings("ignore", category=FutureWarning)
ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print(f"torch {torch.__version__} | timm {timm.__version__} | device {DEVICE} | "
      f"GPUs visible: {torch.cuda.device_count()} "
      f"{torch.cuda.get_device_name(0) if DEVICE == 'cuda' else ''}")

## Run configuration (edit this cell)

In [ ]:
RUN_MODE = os.environ.get("PAD_RUN_MODE", "quick")   # "quick" = smoke test (minutes) | "full" = paper protocol

# Which seen domains to train, in order.
#   "kaggle-4": Market-1501 -> CUHK-SYSU -> DukeMTMC-reID -> CUHK03 (all on Kaggle; AKA order-1 without MSMT17,
#               whose images are not on Kaggle)
#   "paper-5" : Market-1501 -> CUHK-SYSU -> DukeMTMC-reID -> MSMT17 -> CUHK03 (AKA order-1, as in the paper;
#               MSMT17 must be requested from its authors and uploaded as a private dataset)
PROTOCOL = os.environ.get("PAD_PROTOCOL", "kaggle-4")
TRAIN_ORDER = {
    "kaggle-4": ["market1501", "cuhksysu", "dukemtmcreid", "cuhk03"],
    "paper-5": ["market1501", "cuhksysu", "dukemtmcreid", "msmt17", "cuhk03"],
}[PROTOCOL]
SKIP_MISSING_DOMAINS = True  # True: leave out any domain whose data isn't attached (with a warning) instead of stopping
RUN_UNSEEN_EVAL = True      # evaluate the final model on the unseen datasets that are available
# Unseen test sets that are publicly downloadable (no request form) are fetched automatically:
# VIPeR, GRID, i-LIDS (original university links, with an Internet Archive fallback) and SenseReID (Google Drive).
# CUHK01/CUHK02 need a request form and PRID2011 has no stable direct file link, so attach those yourself if you have them.
DOWNLOAD_PUBLIC_UNSEEN = True
PRID_ZIP_URL = ""           # optional: paste the prid_2011.zip link from the TU Graz PRID page to fetch PRID too
RESUME = True               # skip domains whose checkpoint already exists (in OUT_DIR or in an attached old output)

# Faithfulness switch for intermediate evaluations (see the report, Sec. "Implementation observations"):
# the official test script rebuilds VA-Prompt "active slots" from non-zero prompt weights, which counts ALL 36
# (randomly initialised) slots even after step 1. True = do exactly what the official code does.
OFFICIAL_EVAL_SLOT_RECONSTRUCTION = True

# Paths (Kaggle defaults; can be overridden with env vars for local runs)
INPUT_ROOT = os.environ.get("PAD_INPUT_ROOT", "/kaggle/input")
WORK_DIR = os.environ.get("PAD_WORK_DIR", "/kaggle/working")
_tmp_base = "/kaggle/temp" if osp.isdir("/kaggle/temp") else "/tmp"
DATA_ROOT = os.environ.get("PAD_DATA_ROOT", osp.join(_tmp_base, "pad_data"))   # symlinks + converted data
OUT_DIR = osp.join(WORK_DIR, "PAD_outputs")                                     # checkpoints, logs, csv
CLIP_CACHE = os.environ.get("PAD_CLIP_CACHE", osp.join(_tmp_base, "clip_cache"))
os.makedirs(DATA_ROOT, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CLIP_CACHE, exist_ok=True)

# Manually point a dataset to a folder if auto-discovery fails, e.g.
# DATA_OVERRIDES = {"market1501": "/kaggle/input/market-1501/Market-1501-v15.09.15"}
DATA_OVERRIDES = {}

NUM_WORKERS = min(4, os.cpu_count() or 2)   # Kaggle has 4 vCPUs (official config uses 8/12; only affects speed)

# Quick mode: tiny schedule so the whole pipeline runs end-to-end in a few minutes.
QUICK = dict(stage1_epochs=1, stage2_epochs=1, max_iters_per_epoch=10, eval_max_query=300, eval_max_gallery=3000)
if RUN_MODE != "full":
    print("!" * 100 + "\n!! RUN_MODE = \"quick\": SMOKE TEST ONLY (1 short epoch, subsampled eval). Numbers are NOT results. "
          "Set RUN_MODE = \"full\" for the real experiment.\n" + "!" * 100)
print("PROTOCOL =", PROTOCOL, "| order:", " -> ".join(TRAIN_ORDER), "| RUN_MODE =", RUN_MODE, "| DATA_ROOT =", DATA_ROOT, "| OUT_DIR =", OUT_DIR)

## Hyper-parameters: a 1:1 port of the official `configs/pad.yml`
Shared settings come first. `DOMAINS` holds the per-domain overrides, merged exactly like `pad/config.py`.

In [ ]:
class CN(dict):
    """Tiny attribute-dict so the ported code can keep `cfg.SOLVER.STAGE1.MAX_EPOCHS` syntax."""
    def __getattr__(self, k):
        try:
            return self[k]
        except KeyError:
            raise AttributeError(k)

    def __setattr__(self, k, v):
        self[k] = v

    @staticmethod
    def wrap(d):
        return CN({k: CN.wrap(v) if isinstance(v, dict) else v for k, v in d.items()})


BASE_CFG = {
    "MODEL": {"NAME": "ViT-B-16", "STRIDE_SIZE": [16, 16], "ID_LOSS_WEIGHT": 0.25, "TRIPLET_LOSS_WEIGHT": 1.0,
              "I2T_LOSS_WEIGHT": 1.0, "LABEL_SMOOTH": True},
    "INPUT": {"SIZE_TRAIN": [256, 128], "SIZE_TEST": [256, 128], "PROB": 0.5, "RE_PROB": 0.5, "PADDING": 10,
              "PIXEL_MEAN": [0.5, 0.5, 0.5], "PIXEL_STD": [0.5, 0.5, 0.5]},
    "DATALOADER": {"SAMPLER": "softmax_triplet", "NUM_INSTANCE": 4, "NUM_WORKERS": 8},
    "DATASETS": {"NAMES": ("market1501",), "ROOT_DIR": DATA_ROOT},
    "SOLVER": {
        "SEED": 42, "MARGIN": 0.3,
        "STAGE1": {"IMS_PER_BATCH": 64, "OPTIMIZER": "Adam", "BASE_LR": 0.00035, "WEIGHT_DECAY": 1e-4,
                   "MAX_EPOCHS": 12, "LOG_PERIOD": 50, "WARMUP_EPOCHS": 3, "WARMUP_LR_INIT": 1e-5, "LR_MIN": 1e-6},
        "STAGE2": {"IMS_PER_BATCH": 64, "OPTIMIZER": "Adam", "BASE_LR": 5e-6, "WEIGHT_DECAY": 1e-4,
                   "MAX_EPOCHS": 25, "LOG_PERIOD": 50, "STEPS": [14, 22], "GAMMA": 0.1, "WARMUP_ITERS": 10,
                   "WARMUP_FACTOR": 0.1, "WARMUP_METHOD": "linear"},
    },
    "TEST": {"IMS_PER_BATCH": 128, "FEAT_NORM": "yes", "NECK_FEAT": "before"},
    "TA_PROMPT": {"N_CTX": 4, "N_CLS_CTX": 4},
    "VA_PROMPT": {"ENABLE": True, "G_LEN": 6, "E_LEN": 6, "POOL_SIZE": 36, "TOP_K": 4, "EMBEDDING_KEY": "mean",
                  "KEY_LEARNABLE": True, "STRIP_AFTER_BLOCK": True, "NEW_SLOTS_PER_DOMAIN": 8},
    "UNFREEZE": {"LAST_BLOCKS": 4},
    "OPT": {"PROMPT_LR": 0.005, "PROMPT_WD": 0.0},
    "DISTILL": {"ENABLE": True, "LAMBDA_FEAT": 0.5, "LAMBDA_LOGIT": 0.5, "TEMP": 4.0, "EMA_MOMENTUM": 0.997},
    "TEXTKD": {"ENABLE": True, "LAMBDA_LOGIT": 0.5, "TEMP": 0.07, "LOGITS_SCALE": 7.0, "NEG_PER_BATCH": 256},
    "EVAL": {"SPLIT_ID": 0, "SEED": 123, "DIRECTION": "auto"},
    "OUTPUT_DIR": "",
}

DOMAIN_OVERRIDES = {
    # First domain: no teacher -> full visual-branch tuning, longer schedule, KD off.
    "market1501": {"SOLVER": {"STAGE1": {"MAX_EPOCHS": 20}, "STAGE2": {"MAX_EPOCHS": 30, "STEPS": [18, 26]}},
                   "DISTILL": {"ENABLE": False}, "TEXTKD": {"ENABLE": False}, "OPT": {"PROMPT_LR": 0.002}},
    "cuhksysu": {},
    "dukemtmcreid": {},
    # Larger dataset -> slightly longer schedule.
    "msmt17": {"DATALOADER": {"NUM_WORKERS": 12},
               "SOLVER": {"STAGE1": {"MAX_EPOCHS": 15}, "STAGE2": {"MAX_EPOCHS": 32, "STEPS": [18, 28]}}},
    # Small dataset -> smaller batch, stronger erasing, fewer new prompt slots.
    "cuhk03": {"INPUT": {"RE_PROB": 0.6},
               "SOLVER": {"STAGE1": {"IMS_PER_BATCH": 48, "MAX_EPOCHS": 10},
                          "STAGE2": {"IMS_PER_BATCH": 48, "MAX_EPOCHS": 28, "STEPS": [16, 24]}},
               "VA_PROMPT": {"NEW_SLOTS_PER_DOMAIN": 4}},
}


def _deep_merge(base, override):
    out = copy.deepcopy(base)
    for k, v in override.items():
        if k in out and isinstance(out[k], dict) and isinstance(v, dict):
            out[k] = _deep_merge(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out


def build_cfg(domain_name, is_first_domain):
    """Shared block + that domain's overrides (== pad.config.load_cfg_from_yaml)."""
    raw = _deep_merge(BASE_CFG, DOMAIN_OVERRIDES.get(domain_name, {}))
    raw["DATASETS"]["NAMES"] = (domain_name,)
    if not is_first_domain and domain_name == "market1501":
        # Market as a *later* domain (other orders) should use the shared schedule + KD like the others.
        raw = _deep_merge(BASE_CFG, {}); raw["DATASETS"]["NAMES"] = (domain_name,)
    cfg = CN.wrap(raw)
    cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS
    if RUN_MODE == "quick":
        cfg.SOLVER.STAGE1.MAX_EPOCHS = QUICK["stage1_epochs"]
        cfg.SOLVER.STAGE2.MAX_EPOCHS = QUICK["stage2_epochs"]
        cfg.SOLVER.STAGE1.LOG_PERIOD = cfg.SOLVER.STAGE2.LOG_PERIOD = 5
    return cfg


print(json.dumps(build_cfg("msmt17", False)["SOLVER"], indent=1))

## Data discovery and preparation
Kaggle mounts each dataset under `/kaggle/input/<slug>/...` with a layout chosen by the uploader. This cell walks
`/kaggle/input`, recognises each dataset by its **folder signature**, and builds the exact layout the official
loaders expect under `DATA_ROOT`, using symlinks so nothing is copied:

```
DATA_ROOT/market1501/{bounding_box_train,query,bounding_box_test}
DATA_ROOT/dukemtmcreid/{...}
DATA_ROOT/msmt17/{train,test,list_train.txt,list_val.txt,list_query.txt,list_gallery.txt}
DATA_ROOT/cuhksysu/{...}   <- converted: {pid}_c1_{idx}.jpg (query uses c2, see below)
DATA_ROOT/cuhk03/{...}     <- converted from cuhk-03.mat with the CUHK03-NP detected split
DATA_ROOT/{cuhk01,cuhk02,grid,ilids,prid,sensereid,viper}/...   (optional unseen sets)
```

In [ ]:
def _list_dirs(root, max_depth=11):   # Kaggle now mounts at /kaggle/input/datasets/<user>/<slug>/...
    """All directories under root (depth-limited, follows Kaggle's read-only mounts)."""
    out = []
    root = osp.abspath(root)
    base_depth = root.rstrip("/").count("/")
    for cur, dirs, files in os.walk(root, followlinks=True):
        depth = cur.rstrip("/").count("/") - base_depth
        if depth >= max_depth:
            dirs[:] = []
        dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("__MACOSX",)]
        out.append(cur)
    return out


ALL_INPUT_DIRS = _list_dirs(INPUT_ROOT) if osp.isdir(INPUT_ROOT) else []
print(f"scanned {len(ALL_INPUT_DIRS)} directories under {INPUT_ROOT}")


def _has_images(d, exts=(".jpg", ".jpeg", ".png", ".bmp"), min_count=1):
    try:
        n = 0
        for f in os.scandir(d):
            if f.name.lower().endswith(exts):
                n += 1
                if n >= min_count:
                    return True
    except (FileNotFoundError, NotADirectoryError, PermissionError):
        pass
    return False


def _find(pred, keyword=None):
    """Shallowest input dir satisfying pred (and containing keyword in its path, if given)."""
    cands = [d for d in ALL_INPUT_DIRS if (keyword is None or keyword in d.lower()) and pred(d)]
    cands.sort(key=lambda p: (p.count("/"), len(p)))
    return cands[0] if cands else None


def _market_style(d):
    return all(_has_images(osp.join(d, s)) for s in ("bounding_box_train", "query", "bounding_box_test"))


def _symlink(src, dst):
    if osp.lexists(dst):
        if osp.islink(dst):
            os.unlink(dst)
        else:
            return
    os.makedirs(osp.dirname(dst), exist_ok=True)
    os.symlink(src, dst)


def _link_market_style(src, name):
    dst = osp.join(DATA_ROOT, name)
    os.makedirs(dst, exist_ok=True)
    for s in ("bounding_box_train", "query", "bounding_box_test"):
        _symlink(osp.join(src, s), osp.join(dst, s))
    return dst


PREPARED = {}

# ---------- Market-1501 / DukeMTMC-reID: plain symlinks ---------------------------------------------------------------
for name, kw in (("market1501", "market"), ("dukemtmcreid", "duke")):
    src = DATA_OVERRIDES.get(name) or _find(_market_style, kw)
    if src:
        PREPARED[name] = _link_market_style(src, name)
        print(f"[data] {name:13s} <- {src}")

# ---------- MSMT17 (V1: train/test, V2: mask_train_v2/mask_test_v2) ------------------------------------------------------
def _msmt_root(d):
    return osp.isfile(osp.join(d, "list_train.txt")) and (
        osp.isdir(osp.join(d, "train")) or osp.isdir(osp.join(d, "mask_train_v2")))


src = DATA_OVERRIDES.get("msmt17") or _find(_msmt_root, "msmt")
if src:
    dst = osp.join(DATA_ROOT, "msmt17"); os.makedirs(dst, exist_ok=True)
    tr = "train" if osp.isdir(osp.join(src, "train")) else "mask_train_v2"
    te = "test" if osp.isdir(osp.join(src, "test")) else "mask_test_v2"
    _symlink(osp.join(src, tr), osp.join(dst, "train")); _symlink(osp.join(src, te), osp.join(dst, "test"))
    for f in ("list_train.txt", "list_val.txt", "list_query.txt", "list_gallery.txt"):
        _symlink(osp.join(src, f), osp.join(dst, f))
    PREPARED["msmt17"] = dst
    print(f"[data] msmt17        <- {src}")
else:
    src = _find(_market_style, "msmt")   # some uploads ship MSMT17 already in Market format
    if src:
        PREPARED["msmt17"] = _link_market_style(src, "msmt17")
        print(f"[data] msmt17 (market-style) <- {src}")


# ---------- CUHK-SYSU -> cuhksysu4reid -> PAD naming -----------------------------------------------------------------
def _crop_save(img_path, box_xyxy, out_path):
    im = Image.open(img_path).convert("RGB")
    im.crop(tuple(int(round(v)) for v in box_xyxy)).save(out_path)


def _mat_str(x):
    """Unwrap a MATLAB char/cell value loaded by scipy into a Python str."""
    while isinstance(x, np.ndarray):
        if x.size == 0:
            return ""
        x = x.ravel()[0]
    return str(x)


def _mat_box(x):
    """MATLAB [x y w h] -> float array [x1 y1 x2 y2]; empty array if the field is empty."""
    b = np.asarray(x, dtype=object)
    while b.dtype == object and b.size == 1 and isinstance(b.ravel()[0], np.ndarray):
        b = b.ravel()[0]
    b = np.asarray(b, dtype=np.float64).ravel()
    if b.size < 4:
        return np.zeros(0)
    b = b[:4].copy(); b[2:] += b[:2]
    return b


def _find_file(name):
    for d in ALL_INPUT_DIRS:
        if osp.isfile(osp.join(d, name)):
            return osp.join(d, name)
    return None


def convert_cuhksysu_raw(train_mat_path, test_mat_path, img_dir, out_dir):
    """Raw CUHK-SYSU person-search release -> ReID crops with the AKA `cuhksysu4reid` splits:
    train = all boxes of the 5,532 Train.mat identities; query = the 2,900 TestG50 query boxes;
    gallery = the boxes of the query person found in its 50-image gallery (images without the person are skipped).
    Port of LifelongReID/lreid/datasets/cuhksysu.py with defensive unwrapping of MATLAB fields."""
    from scipy.io import loadmat
    tmp = out_dir + ".partial"
    shutil.rmtree(tmp, ignore_errors=True)
    for sub in ("bounding_box_train", "query", "bounding_box_test"):
        os.makedirs(osp.join(tmp, sub), exist_ok=True)
    train_mat = loadmat(train_mat_path)["Train"].squeeze()
    test_mat = loadmat(test_mat_path)["TestG50"].squeeze()
    k = 0
    for item in train_mat:                                            # 5,532 identities
        rec = item[0, 0] if isinstance(item, np.ndarray) and item.dtype.names is None else item
        rec = rec if not isinstance(rec, np.ndarray) else rec.ravel()[0]
        pid = int(_mat_str(rec[0])[1:])                               # 'p01234' -> 1234
        for scene in np.asarray(rec[2]).ravel():
            box = _mat_box(scene[1])
            if box.size:
                _crop_save(osp.join(img_dir, _mat_str(scene[0])), box,
                           osp.join(tmp, "bounding_box_train", f"{pid:05d}_c1_{k:07d}.jpg")); k += 1
    for query, gallery in zip(test_mat["Query"], test_mat["Gallery"]):   # 2,900 queries
        pid = int(_mat_str(query["idname"])[1:])
        _crop_save(osp.join(img_dir, _mat_str(query["imname"])), _mat_box(query["idlocate"]),
                   osp.join(tmp, "query", f"{pid:05d}_c2_{k:07d}.jpg")); k += 1
        for g in np.asarray(gallery).ravel():
            gbox = _mat_box(g["idlocate"])
            if gbox.size == 0:                                        # gallery image without the target person
                continue
            _crop_save(osp.join(img_dir, _mat_str(g["imname"])), gbox,
                       osp.join(tmp, "bounding_box_test", f"{pid:05d}_c1_{k:07d}.jpg")); k += 1
    shutil.rmtree(out_dir, ignore_errors=True)
    os.replace(tmp, out_dir)


def relink_cuhksysu4reid(src, out_dir):
    """cuhksysu4reid/{train,query,gallery}/{pid}_s{scene}_{idx}_{hard}.jpg -> PAD naming via symlinks.
    NOTE: the README says 'renamed to {pid:04d}_c1_{idx:05d}.jpg'. If query AND gallery are all camera 1, the standard
    evaluator (which drops same-id/same-camera gallery hits) would discard every true match, so the query split is
    given camera 2 here (exactly what the AKA/PTKP loaders do: query cam=1, gallery cam=0)."""
    pat = re.compile(r"([-\d]+)_s([-\d]+)_([-\d]+)")
    for sub, dsub, cam in (("train", "bounding_box_train", 1), ("query", "query", 2), ("gallery", "bounding_box_test", 1)):
        os.makedirs(osp.join(out_dir, dsub), exist_ok=True)
        for i, p in enumerate(sorted(glob.glob(osp.join(src, sub, "*.jpg")))):
            m = pat.search(osp.basename(p))
            if m is None:
                continue
            _symlink(p, osp.join(out_dir, dsub, f"{int(m.group(1)):05d}_c{cam}_{i:07d}.jpg"))


sysu_out = osp.join(DATA_ROOT, "cuhksysu")
if _market_style(sysu_out):
    PREPARED["cuhksysu"] = sysu_out
    print(f"[data] cuhksysu      <- (already prepared) {sysu_out}")
else:
    src4 = DATA_OVERRIDES.get("cuhksysu") or _find(
        lambda d: all(_has_images(osp.join(d, s)) for s in ("train", "query", "gallery")), "sysu")
    src_ms = _find(_market_style, "sysu")
    sysu_train_mat, sysu_test_mat = _find_file("Train.mat"), _find_file("TestG50.mat")
    sysu_img_dir = _find(lambda d: osp.basename(d) == "SSM" and _has_images(d, (".jpg",)))
    if src4:
        relink_cuhksysu4reid(src4, sysu_out); PREPARED["cuhksysu"] = sysu_out
        print(f"[data] cuhksysu      <- cuhksysu4reid {src4}")
    elif src_ms:
        PREPARED["cuhksysu"] = _link_market_style(src_ms, "cuhksysu")
        print(f"[data] cuhksysu      <- market-style {src_ms} (check that query/gallery cameras differ!)")
    elif sysu_train_mat and sysu_test_mat and sysu_img_dir:
        print(f"[data] cuhksysu      converting raw person-search release (images {sysu_img_dir}) ~5-10 min ...")
        try:
            convert_cuhksysu_raw(sysu_train_mat, sysu_test_mat, sysu_img_dir, sysu_out)
            PREPARED["cuhksysu"] = sysu_out
        except Exception as e:  # noqa: BLE001
            print(f"!! cuhksysu conversion failed ({type(e).__name__}: {e}); the domain will be left out.")
    elif _find(lambda d: True, "sysu"):
        print("!! a CUHK-SYSU-like input is attached but its layout was not recognised "
              "(need Image/SSM + annotation/test/train_test/{Train,TestG50}.mat, or cuhksysu4reid). See the tree below.")


# ---------- CUHK03 (cuhk-03.mat -> CUHK03-NP detected) -----------------------------------------------------------------
CUHK03_NP_URL = ("https://raw.githubusercontent.com/zhunzhong07/person-re-ranking/master/evaluation/data/CUHK03/"
                 "cuhk03_new_protocol_config_detected.mat")   # dataset split file (Zhong et al. CVPR'17), not PAD code


def convert_cuhk03(mat_path, split_path, out_dir, mode="detected"):
    """Same output as the official tool/convert_cuhk03.py ({pid:04d}_c{1..10}_{idx:05d}.jpg, camid = 2*pair-1 for
    view A and 2*pair for view B), but reads the v7.3 .mat with h5py and maps images by the protocol's file list."""
    import h5py
    from scipy.io import loadmat
    f = h5py.File(mat_path, "r")
    pairs = [f[ref][:].T for ref in f[mode][0]]           # 5 camera pairs, each (num_ids, 10) object refs
    sp = loadmat(split_path)
    def _s(x):   # unwrap MATLAB cell -> str
        while isinstance(x, np.ndarray):
            x = x.ravel()[0]
        return str(x)
    files = [_s(x) for x in sp["filelist"]]
    labels = sp["labels"].ravel().astype(int)
    for key, sub in (("train_idx", "bounding_box_train"), ("query_idx", "query"), ("gallery_idx", "bounding_box_test")):
        os.makedirs(osp.join(out_dir, sub), exist_ok=True)
        for i, k in enumerate(sp[key].ravel().astype(int) - 1):
            pair, pid_in_pair, view, img_id = map(int, osp.splitext(files[k])[0].split("_"))
            img = f[pairs[pair - 1][pid_in_pair - 1, img_id - 1]][:].T      # (H, W, 3) uint8
            camid = 2 * pair - (1 if view == 1 else 0)
            Image.fromarray(img.astype(np.uint8)).save(
                osp.join(out_dir, sub, f"{labels[k]:04d}_c{camid}_{i + 1:05d}.jpg"), quality=95)
    f.close()


c03_out = osp.join(DATA_ROOT, "cuhk03")
if _market_style(c03_out):
    PREPARED["cuhk03"] = c03_out
    print(f"[data] cuhk03        <- (already prepared) {c03_out}")
elif DATA_OVERRIDES.get("cuhk03"):
    PREPARED["cuhk03"] = _link_market_style(DATA_OVERRIDES["cuhk03"], "cuhk03")
else:
    mats = [osp.join(d, "cuhk-03.mat") for d in ALL_INPUT_DIRS if osp.isfile(osp.join(d, "cuhk-03.mat"))]
    if mats:
        splits = [osp.join(d, "cuhk03_new_protocol_config_detected.mat") for d in ALL_INPUT_DIRS
                  if osp.isfile(osp.join(d, "cuhk03_new_protocol_config_detected.mat"))]
        split_path = splits[0] if splits else osp.join(DATA_ROOT, "cuhk03_new_protocol_config_detected.mat")
        if not splits and not osp.isfile(split_path):
            print("[data] cuhk03: downloading CUHK03-NP split file ...")
            urllib.request.urlretrieve(CUHK03_NP_URL, split_path)
        print(f"[data] cuhk03        converting {mats[0]} (CUHK03-NP detected, ~3 min) ...")
        convert_cuhk03(mats[0], split_path, c03_out); PREPARED["cuhk03"] = c03_out
    else:
        src = _find(_market_style, "cuhk03")
        if src:
            PREPARED["cuhk03"] = _link_market_style(src, "cuhk03")
            print(f"[data] cuhk03        <- market-style {src}")

### Public unseen test sets (automatic download)
None of the paper's 7 unseen datasets are on Kaggle. Four are publicly downloadable without a request form, so this
cell fetches them directly (Internet must be on). Each one has several sources tried in order: the original
university link that torchreid/Open-ReID use, then an Internet Archive (Wayback Machine) copy of the same file.
Archives are integrity-checked (and VIPeR's MD5 is verified) and extracted safely, and any source that fails is
skipped with a message, so the rest of the notebook always runs.

| Set | Status | Source |
|---|---|---|
| VIPeR | auto | users.soe.ucsc.edu/~manduchi/VIPeR.v1.0.zip (MD5 `1c2d9fc1…`) |
| GRID | auto | personal.ie.cuhk.edu.hk/~ccloy/files/datasets/underground_reid.zip |
| i-LIDS (QMUL pedestrian) | auto | eecs.qmul.ac.uk/~jason/data/i-LIDS_Pedestrian.tgz |
| SenseReID | auto | Google Drive file `0B56OfSrVI8hubVJLTzkwV2VaOWM` (via `gdown`) |
| PRID2011 | optional | paste the `prid_2011.zip` link from the TU Graz page into `PRID_ZIP_URL`, or attach it |
| CUHK01 / CUHK02 | attach only | request form at CUHK; upload as a private Kaggle dataset if you have them |

In [ ]:
import tarfile, urllib.parse, zipfile

UNSEEN_DL_ROOT = osp.join(_tmp_base, "pad_unseen_downloads")
_WB = "https://web.archive.org/web/2020id_/"   # Wayback "id_" = the original file bytes, nearest snapshot to 2020
UNSEEN_PUBLIC_SOURCES = {
    "viper": {"urls": ["http://users.soe.ucsc.edu/~manduchi/VIPeR.v1.0.zip",
                       _WB + "http://users.soe.ucsc.edu/~manduchi/VIPeR.v1.0.zip"],
              "md5": "1c2d9fc1cc800332567a0da25a1ce68c", "file": "VIPeR.v1.0.zip"},
    "grid": {"urls": ["http://personal.ie.cuhk.edu.hk/~ccloy/files/datasets/underground_reid.zip",
                      _WB + "http://personal.ie.cuhk.edu.hk/~ccloy/files/datasets/underground_reid.zip"],
             "file": "underground_reid.zip"},
    "ilids": {"urls": ["http://www.eecs.qmul.ac.uk/~jason/data/i-LIDS_Pedestrian.tgz",
                       _WB + "http://www.eecs.qmul.ac.uk/~jason/data/i-LIDS_Pedestrian.tgz"],
              "file": "i-LIDS_Pedestrian.tgz"},
    "sensereid": {"gdrive": "0B56OfSrVI8hubVJLTzkwV2VaOWM", "file": "SenseReID.zip"},
}
if PRID_ZIP_URL:
    UNSEEN_PUBLIC_SOURCES["prid"] = {"urls": [PRID_ZIP_URL], "file": "prid_2011.zip"}
if os.environ.get("PAD_UNSEEN_SOURCES_JSON"):   # testing hook: override the source list
    UNSEEN_PUBLIC_SOURCES = json.loads(os.environ["PAD_UNSEEN_SOURCES_JSON"])

# Folder signatures used to tell whether a set is already available (attached input or earlier download).
UNSEEN_SIGNATURES = {
    "viper": lambda d: osp.isdir(osp.join(d, "cam_a")) and osp.isdir(osp.join(d, "cam_b")) and _has_images(osp.join(d, "cam_a"), (".bmp",)),
    "grid": lambda d: osp.isdir(osp.join(d, "probe")) and osp.isfile(osp.join(d, "features_and_partitions.mat")),
    "ilids": lambda d: osp.basename(d) == "Persons" and _has_images(d, (".jpg",)),
    "sensereid": lambda d: osp.isdir(osp.join(d, "test_probe")) and osp.isdir(osp.join(d, "test_gallery")),
    "prid": lambda d: osp.basename(d) == "single_shot" and osp.isdir(osp.join(d, "cam_a")),
}


def _safe_extract(archive, dest):
    """Extract only regular files/dirs that stay inside dest (archives from the web are untrusted)."""
    dest_real = osp.realpath(dest)
    os.makedirs(dest, exist_ok=True)

    def _inside(name):
        target = osp.realpath(osp.join(dest, name))
        return target == dest_real or target.startswith(dest_real + os.sep)

    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            for m in z.infolist():
                if _inside(m.filename):
                    z.extract(m, dest)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as t:
            members = [m for m in t.getmembers() if (m.isfile() or m.isdir()) and _inside(m.name)]
            t.extractall(dest, members=members)
    else:
        raise RuntimeError("not a zip/tar archive (probably an HTML error page)")


def _download_url(url, out, timeout=120):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (PAD reproduction notebook)"})
    with urllib.request.urlopen(req, timeout=timeout) as r, open(out + ".part", "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    os.replace(out + ".part", out)


def _wayback_captures(url, limit=3):
    """Ask the Internet Archive index for saved copies (HTTP 200) of `url`; newest first."""
    q = ("https://web.archive.org/cdx/search/cdx?output=json&fl=timestamp,original&filter=statuscode:200"
         f"&limit=-{limit}&url=" + urllib.parse.quote(url.split("://", 1)[-1], safe=""))
    try:
        req = urllib.request.Request(q, headers={"User-Agent": "Mozilla/5.0 (PAD reproduction notebook)"})
        with urllib.request.urlopen(req, timeout=60) as r:
            rows = json.loads(r.read().decode() or "[]")
        return [f"https://web.archive.org/web/{ts}id_/{orig}" for ts, orig in rows[1:]][::-1]
    except Exception:  # noqa: BLE001
        return []


def fetch_public_unseen(name, spec):
    dest = osp.join(UNSEEN_DL_ROOT, name)
    done_flag = osp.join(dest, ".extracted")
    if osp.isfile(done_flag):
        return "already downloaded"
    os.makedirs(dest, exist_ok=True)
    archive = osp.join(UNSEEN_DL_ROOT, spec["file"])
    errors = []
    attempts = [("gdrive", spec["gdrive"])] if spec.get("gdrive") else []
    attempts += [("url", u) for u in spec.get("urls", [])]
    looked_up = False
    i = 0
    while i < len(attempts):
        kind, src = attempts[i]; i += 1
        try:
            if kind == "gdrive":
                try:
                    import gdown
                except ImportError:
                    import subprocess
                    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
                    import gdown
                if gdown.download(id=src, output=archive, quiet=True) is None:
                    raise RuntimeError("gdown returned nothing (quota or permissions)")
            else:
                _download_url(src, archive)
            if spec.get("md5") and hashlib.md5(open(archive, "rb").read()).hexdigest() != spec["md5"]:
                raise RuntimeError("MD5 mismatch")
            _safe_extract(archive, dest)
            open(done_flag, "w").write(src)
            os.remove(archive)
            return f"ok <- {src}"
        except Exception as e:  # noqa: BLE001
            errors.append(f"{src[:70]}...: {type(e).__name__}: {str(e)[:80]}")
            if osp.exists(archive):
                os.remove(archive)
        if i == len(attempts) and not looked_up:          # all fixed sources failed -> search the archive index
            looked_up = True
            for u in spec.get("urls", []):
                if "web.archive.org" not in u:
                    attempts += [("url", c) for c in _wayback_captures(u) if ("url", c) not in attempts]
    return "FAILED\n      " + "\n      ".join(errors)


if DOWNLOAD_PUBLIC_UNSEEN and RUN_UNSEEN_EVAL:
    os.makedirs(UNSEEN_DL_ROOT, exist_ok=True)
    for name, spec in UNSEEN_PUBLIC_SOURCES.items():
        if name in UNSEEN_SIGNATURES and _find(UNSEEN_SIGNATURES[name]):
            print(f"[unseen] {name:10s} already attached as a Kaggle input, not downloading")
            continue
        print(f"[unseen] {name:10s} {fetch_public_unseen(name, spec)}")
    ALL_INPUT_DIRS += _list_dirs(UNSEEN_DL_ROOT)   # make downloads visible to the discovery below

In [ ]:
# ---------- unseen-domain sets (optional) ----------------------------------------------------------------------------
def _link_unseen(name, src, rel):
    _symlink(src, osp.join(DATA_ROOT, name, rel)); PREPARED[name] = osp.join(DATA_ROOT, name)
    print(f"[data] {name:13s} <- {src}")


def _try(name, pred, rel, kw=None):
    src = DATA_OVERRIDES.get(name) or _find(pred, kw)
    if src:
        _link_unseen(name, src, rel)


_try("viper", lambda d: osp.basename(d).lower() == "viper" and osp.isdir(osp.join(d, "cam_a")) and osp.isdir(osp.join(d, "cam_b")), "VIPeR")
if "viper" not in PREPARED:
    _try("viper", lambda d: osp.isdir(osp.join(d, "cam_a")) and osp.isdir(osp.join(d, "cam_b"))
         and _has_images(osp.join(d, "cam_a"), (".bmp",)), "VIPeR")
_try("prid", lambda d: osp.basename(d) == "single_shot" and osp.isdir(osp.join(d, "cam_a")), "single_shot")
_try("ilids", lambda d: osp.basename(d) == "i-LIDS_Pedestrian" and osp.isdir(osp.join(d, "Persons")), "i-LIDS_Pedestrian")
if "ilids" not in PREPARED:
    _try("ilids", lambda d: osp.basename(d) == "Persons" and _has_images(d, (".jpg",)), "i-LIDS_Pedestrian/Persons", "lids")
_try("cuhk01", lambda d: osp.basename(d).lower() == "campus" and _has_images(d, (".png",)), "campus")
_try("cuhk02", lambda d: osp.isdir(osp.join(d, "P5", "cam1")) and osp.isdir(osp.join(d, "P5", "cam2")), "Dataset")
_try("sensereid", lambda d: osp.isdir(osp.join(d, "test_probe")) and osp.isdir(osp.join(d, "test_gallery")), "SenseReID")
src = DATA_OVERRIDES.get("grid") or _find(lambda d: osp.isdir(osp.join(d, "probe")) and osp.isdir(osp.join(d, "gallery"))
                                          and osp.isfile(osp.join(d, "features_and_partitions.mat")))
if src:
    for s in ("probe", "gallery", "features_and_partitions.mat"):
        _symlink(osp.join(src, s), osp.join(DATA_ROOT, "grid", s))
    PREPARED["grid"] = osp.join(DATA_ROOT, "grid"); print(f"[data] grid          <- {src}")

def _dataset_roots():
    """Top folder of every attached Kaggle dataset (/kaggle/input/<slug> or /kaggle/input/datasets/<user>/<slug>)."""
    roots = set()
    for d in ALL_INPUT_DIRS:
        rel = osp.relpath(d, INPUT_ROOT)
        if rel.startswith(".."):
            continue
        parts = rel.split(os.sep)
        if parts[0] == "datasets" and len(parts) >= 3:
            roots.add(osp.join(INPUT_ROOT, *parts[:3]))
        elif parts[0] not in (".", "datasets") and len(parts) >= 1:
            roots.add(osp.join(INPUT_ROOT, parts[0]))
    return sorted(roots)


def _print_tree(root, depth=3, max_items=6, _lvl=0):
    try:
        entries = sorted(os.scandir(root), key=lambda e: (not e.is_dir(), e.name))
    except OSError:
        return
    dirs = [e for e in entries if e.is_dir()]
    files = [e for e in entries if not e.is_dir()]
    pad = "    " * (_lvl + 1)
    for e in dirs[:max_items]:
        print(f"{pad}{e.name}/")
        if _lvl + 1 < depth:
            _print_tree(e.path, depth, max_items, _lvl + 1)
    if len(dirs) > max_items:
        print(f"{pad}... (+{len(dirs) - max_items} more folders)")
    if files:
        shown = ", ".join(f.name for f in files[:3])
        print(f"{pad}[{len(files)} files: {shown}{', ...' if len(files) > 3 else ''}]")


_used = set()
for link_root, _, fnames in os.walk(DATA_ROOT):
    for n in os.listdir(link_root):
        q = osp.join(link_root, n)
        if osp.islink(q):
            _used.update({osp.abspath(os.readlink(q)), osp.realpath(q)})
for extra in [globals().get("sysu_train_mat"), globals().get("sysu_img_dir")] + list(globals().get("mats") or []):
    if extra:
        _used.update({osp.abspath(extra), osp.realpath(extra)})
for r in _dataset_roots():
    keys = {osp.abspath(r), osp.realpath(r)}
    if "pad_outputs" in r.lower() or any(u == k or u.startswith(k + os.sep) for u in _used for k in keys):
        continue
    if any(osp.isfile(osp.join(d, "ViT-B-16.pt")) for d in ALL_INPUT_DIRS if d.startswith(r)):
        continue
    print(f"\n?? attached input NOT used (layout not recognised): {r}")
    _print_tree(r)

missing = [d for d in TRAIN_ORDER if d not in PREPARED]
print("\nprepared:", sorted(PREPARED))
if missing and SKIP_MISSING_DOMAINS:
    TRAIN_ORDER = [d for d in TRAIN_ORDER if d in PREPARED]
    print(f"!! not attached, LEFT OUT of the sequence: {missing}")
elif missing:
    print(f"!! MISSING seen domains: {missing}. Attach them (see table above) or set DATA_OVERRIDES. "
          f"Training will stop when it reaches the first missing domain.")
assert TRAIN_ORDER, "no seen-domain dataset found under INPUT_ROOT -- attach the Kaggle datasets listed above"
if TRAIN_ORDER[0] != "market1501":
    print("!! Market-1501 is not the first domain: the first-domain schedule (no KD, full tuning) is applied to "
          f"'{TRAIN_ORDER[0]}' with its own per-domain settings, so this no longer matches the paper's order.")
print("training order:", " -> ".join(TRAIN_ORDER))

## Dataset classes (verbatim port of `pad/data/bases.py`, `seen.py`, `unseen.py`, `sampler.py`)
Each loader returns `(img_path, pid, camid, trackid)` tuples; see the official repo for the protocol notes.

In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True


def read_image(img_path: str) -> Image.Image:
    if not osp.exists(img_path):
        raise IOError(f"{img_path} does not exist")
    while True:
        try:
            return Image.open(img_path).convert("RGB")
        except IOError:
            print(f"[read_image] retrying: {img_path}")


class BaseImageDataset:
    """Common statistics helpers for image-based ReID datasets."""

    def get_imagedata_info(self, data):
        pids, cams, tracks = set(), set(), set()
        for _, pid, camid, trackid in data:
            pids.add(pid); cams.add(camid); tracks.add(trackid)
        return len(pids), len(data), len(cams), len(tracks)

    def print_dataset_statistics(self, train, query, gallery):
        n_tp, n_ti, n_tc, _ = self.get_imagedata_info(train)
        n_qp, n_qi, n_qc, _ = self.get_imagedata_info(query)
        n_gp, n_gi, n_gc, _ = self.get_imagedata_info(gallery)
        print("  ------------------------------------------")
        print("  subset  | # ids | # images | # cameras")
        print("  ------------------------------------------")
        print(f"  train   | {n_tp:5d} | {n_ti:8d} | {n_tc:9d}")
        print(f"  query   | {n_qp:5d} | {n_qi:8d} | {n_qc:9d}")
        print(f"  gallery | {n_gp:5d} | {n_gi:8d} | {n_gc:9d}")
        print("  ------------------------------------------")

    def _finalize(self, train, query, gallery, tag: str, verbose: bool = True):
        self.train = train
        self.query = query
        self.gallery = gallery
        if verbose:
            print(f"=> {tag} loaded")
            self.print_dataset_statistics(train, query, gallery)
        (self.num_train_pids, self.num_train_imgs,
         self.num_train_cams, self.num_train_vids) = self.get_imagedata_info(train)
        self.num_query_pids, _, _, _ = self.get_imagedata_info(query)
        self.num_gallery_pids, _, _, _ = self.get_imagedata_info(gallery)
        # Unseen loaders have no training split; clamp so PADModel can still
        # be constructed safely at inference time.
        if self.num_train_pids <= 0:
            self.num_train_pids = 1


class ImageDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_path, pid, camid, trackid = self.data[index]
        img = read_image(img_path)
        if self.transform is not None:
            img = self.transform(img)
        return img, pid, camid, trackid, img_path.split("/")[-1]


class Market1501(BaseImageDataset):
    """Standard Market-1501 (bounding_box_train / query / bounding_box_test)."""

    dataset_dir = "market1501"

    def __init__(self, root: str = "", verbose: bool = True, pid_begin: int = 0, **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.train_dir = osp.join(self.dataset_dir, "bounding_box_train")
        self.query_dir = osp.join(self.dataset_dir, "query")
        self.gallery_dir = osp.join(self.dataset_dir, "bounding_box_test")
        self.pid_begin = pid_begin
        for d in (self.dataset_dir, self.train_dir, self.query_dir, self.gallery_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        train = self._process_dir(self.train_dir, relabel=True)
        query = self._process_dir(self.query_dir, relabel=False)
        gallery = self._process_dir(self.gallery_dir, relabel=False)
        self._finalize(train, query, gallery, "Market-1501", verbose)

    def _process_dir(self, dir_path: str, relabel: bool = False):
        img_paths = sorted(glob.glob(osp.join(dir_path, "*.jpg")))
        pattern = re.compile(r"([-\d]+)_c(\d)")
        pids = {int(pattern.search(p).group(1)) for p in img_paths
                if int(pattern.search(p).group(1)) != -1}
        pid2label = {pid: lab for lab, pid in enumerate(sorted(pids))}
        data = []
        for p in img_paths:
            pid, camid = map(int, pattern.search(p).groups())
            if pid == -1:
                continue
            assert 0 <= pid <= 1501 and 1 <= camid <= 6
            camid -= 1
            if relabel:
                pid = pid2label[pid]
            data.append((p, self.pid_begin + pid, camid, 0))
        return data


class DukeMTMCreID(BaseImageDataset):
    """Standard Duke layout (bounding_box_train / query / bounding_box_test)."""

    dataset_dir = "dukemtmcreid"

    def __init__(self, root: str = "", verbose: bool = True, pid_begin: int = 0, **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.train_dir = osp.join(self.dataset_dir, "bounding_box_train")
        self.query_dir = osp.join(self.dataset_dir, "query")
        self.gallery_dir = osp.join(self.dataset_dir, "bounding_box_test")
        self.pid_begin = pid_begin
        for d in (self.dataset_dir, self.train_dir, self.query_dir, self.gallery_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        train = self._process_dir(self.train_dir, relabel=True)
        query = self._process_dir(self.query_dir, relabel=False)
        gallery = self._process_dir(self.gallery_dir, relabel=False)
        self._finalize(train, query, gallery, "DukeMTMC-reID", verbose)

    def _process_dir(self, dir_path: str, relabel: bool = False):
        img_paths = sorted(glob.glob(osp.join(dir_path, "*.jpg")))
        pattern = re.compile(r"([-\d]+)_c(\d)")
        pids = {int(pattern.search(p).group(1)) for p in img_paths}
        pid2label = {pid: lab for lab, pid in enumerate(sorted(pids))}
        data = []
        for p in img_paths:
            pid, camid = map(int, pattern.search(p).groups())
            assert 1 <= camid <= 8
            camid -= 1
            if relabel:
                pid = pid2label[pid]
            data.append((p, self.pid_begin + pid, camid, 0))
        return data


class MSMT17(BaseImageDataset):
    """MSMT17 (train / test folders + list_*.txt annotation files)."""

    dataset_dir = "msmt17"

    def __init__(self, root: str = "", verbose: bool = True, pid_begin: int = 0, **kwargs):
        self.pid_begin = pid_begin
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.train_dir = osp.join(self.dataset_dir, "train")
        self.test_dir = osp.join(self.dataset_dir, "test")
        lt = osp.join(self.dataset_dir, "list_train.txt")
        lv = osp.join(self.dataset_dir, "list_val.txt")
        lq = osp.join(self.dataset_dir, "list_query.txt")
        lg = osp.join(self.dataset_dir, "list_gallery.txt")
        for d in (self.dataset_dir, self.train_dir, self.test_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        train = self._process_dir(self.train_dir, lt) + self._process_dir(self.train_dir, lv)
        query = self._process_dir(self.test_dir, lq)
        gallery = self._process_dir(self.test_dir, lg)
        self._finalize(train, query, gallery, "MSMT17", verbose)

    def _process_dir(self, dir_path: str, list_path: str):
        data = []
        with open(list_path, "r") as f:
            for line in f:
                img_name, pid = line.strip().split(" ")
                pid = int(pid)
                camid = int(img_name.split("_")[2]) - 1
                data.append((osp.join(dir_path, img_name), self.pid_begin + pid, camid, 0))
        return data


class CUHKSYSU(BaseImageDataset):
    """Filenames follow ``{pid:04d}_c{camid}_{idx:05d}.jpg`` (ReID-style conversion)."""

    dataset_dir = "cuhksysu"

    def __init__(self, root: str = "", verbose: bool = True, pid_begin: int = 0, **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.train_dir = osp.join(self.dataset_dir, "bounding_box_train")
        self.query_dir = osp.join(self.dataset_dir, "query")
        self.gallery_dir = osp.join(self.dataset_dir, "bounding_box_test")
        self.pid_begin = pid_begin
        for d in (self.dataset_dir, self.train_dir, self.query_dir, self.gallery_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        train = self._process_dir(self.train_dir, relabel=True)
        query = self._process_dir(self.query_dir, relabel=False)
        gallery = self._process_dir(self.gallery_dir, relabel=False)
        self._finalize(train, query, gallery, "CUHK-SYSU", verbose)

    def _process_dir(self, dir_path: str, relabel: bool = False):
        img_paths = sorted(glob.glob(osp.join(dir_path, "*.jpg")))
        pattern = re.compile(r"([-\d]+)_c(\d+)")
        pids = set()
        for p in img_paths:
            m = pattern.search(osp.basename(p))
            if m is not None and int(m.group(1)) != -1:
                pids.add(int(m.group(1)))
        pid2label = {pid: lab for lab, pid in enumerate(sorted(pids))}
        data = []
        for p in img_paths:
            m = pattern.search(osp.basename(p))
            if m is None:
                continue
            pid, camid = map(int, m.groups())
            if pid == -1:
                continue
            assert camid >= 1
            camid -= 1
            if relabel:
                pid = pid2label[pid]
            data.append((p, self.pid_begin + pid, camid, 0))
        return data


class CUHK03(BaseImageDataset):
    """CUHK03 new-protocol (detected) layout; camid in [1, 10]."""

    dataset_dir = "cuhk03"

    def __init__(self, root: str = "", verbose: bool = True, pid_begin: int = 0, **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.train_dir = osp.join(self.dataset_dir, "bounding_box_train")
        self.query_dir = osp.join(self.dataset_dir, "query")
        self.gallery_dir = osp.join(self.dataset_dir, "bounding_box_test")
        self.pid_begin = pid_begin
        for d in (self.dataset_dir, self.train_dir, self.query_dir, self.gallery_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        train = self._process_dir(self.train_dir, relabel=True)
        query = self._process_dir(self.query_dir, relabel=False)
        gallery = self._process_dir(self.gallery_dir, relabel=False)
        self._finalize(train, query, gallery, "CUHK03", verbose)

    def _process_dir(self, dir_path: str, relabel: bool = False):
        img_paths = sorted(glob.glob(osp.join(dir_path, "*.jpg")))
        pattern = re.compile(r"([-\d]+)_c(\d+)_")
        pids = set()
        for p in img_paths:
            m = pattern.search(osp.basename(p))
            if m is not None:
                pids.add(int(m.group(1)))
        pid2label = {pid: lab for lab, pid in enumerate(sorted(pids))}
        data = []
        for p in img_paths:
            m = pattern.search(osp.basename(p))
            if m is None:
                continue
            pid, camid = map(int, m.groups())
            assert 1 <= camid <= 10, f"unexpected camid {camid} in {p}"
            camid -= 1
            if relabel:
                pid = pid2label[pid]
            data.append((p, self.pid_begin + pid, camid, 0))
        return data


class LPW_S2(BaseImageDataset):
    """LPW-s2 (pre-converted to Market-style ``<pid>_c<camid>s<sid>_*.jpg``)."""

    dataset_dir = "LPW_s2"

    def __init__(self, root: str = "", verbose: bool = True, pid_begin: int = 0, **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.train_dir = osp.join(self.dataset_dir, "bounding_box_train")
        self.query_dir = osp.join(self.dataset_dir, "query")
        self.gallery_dir = osp.join(self.dataset_dir, "bounding_box_test")
        self.pid_begin = pid_begin
        for d in (self.dataset_dir, self.train_dir, self.query_dir, self.gallery_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        train = self._process_dir(self.train_dir, relabel=True)
        query = self._process_dir(self.query_dir, relabel=False)
        gallery = self._process_dir(self.gallery_dir, relabel=False)
        self._finalize(train, query, gallery, "LPW-s2", verbose)

    def _process_dir(self, dir_path: str, relabel: bool = False):
        img_paths = sorted(glob.glob(osp.join(dir_path, "*.jpg")))
        pattern = re.compile(r"([-\d]+)_c(\d+)s(\d+)")
        pids = set()
        for p in img_paths:
            m = pattern.search(osp.basename(p))
            if m is not None and int(m.group(1)) != -1:
                pids.add(int(m.group(1)))
        pid2label = {pid: lab for lab, pid in enumerate(sorted(pids))}
        data = []
        for p in img_paths:
            m = pattern.search(osp.basename(p))
            if m is None:
                continue
            pid, camid, _ = map(int, m.groups())
            if pid == -1:
                continue
            camid -= 1
            if relabel:
                pid = pid2label[pid]
            data.append((p, self.pid_begin + pid, camid, 0))
        return data


class CUHK01(BaseImageDataset):
    """``<root>/cuhk01/campus/*.png``, filenames ``PPPPCCCC.png``.

    Cams 1-2 form view A, cams 3-4 form view B. Following the standard
    protocol (see e.g. torchreid's ``CUHK01`` loader), identities are
    randomly split in half; only the held-out test half (~486 of 971)
    is evaluated, in both directions (A2B/B2A), averaged over 10 splits
    (20 runs total, matching VIPeR's convention).
    """

    dataset_dir = "cuhk01"

    def __init__(self, root: str = "", verbose: bool = True,
                 split_id: int = 0, seed: int = 123, direction: str = "auto", **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.campus_dir = osp.join(self.dataset_dir, "campus")
        for d in (self.dataset_dir, self.campus_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        view_a, view_b = defaultdict(list), defaultdict(list)
        for p in sorted(glob.glob(osp.join(self.campus_dir, "*.png"))):
            fname = osp.basename(p)
            try:
                pid = int(fname[:4])
                cam = int(fname[4:7]) if fname[4:7].isdigit() else int(fname[4:8])
            except Exception:
                continue
            if cam in (1, 2):
                view_a[pid].append((p, 0, 0))
            elif cam in (3, 4):
                view_b[pid].append((p, 1, 0))

        all_pids = sorted(set(view_a) & set(view_b))
        if not all_pids:
            raise RuntimeError("CUHK01: no identity overlap between view A and view B")

        if direction in ("A2B", "B2A"):
            eval_direction, base_split = direction, split_id
        else:
            eval_direction = "A2B" if split_id % 2 == 0 else "B2A"
            base_split = split_id // 2

        rng = random.Random(seed + base_split)
        shuffled = all_pids[:]
        rng.shuffle(shuffled)
        num_train = int(len(shuffled) * 0.5)
        test_pids = shuffled[num_train:]

        view_first, view_second = (view_a, view_b) if eval_direction == "A2B" else (view_b, view_a)
        query, gallery = [], []
        for pid in test_pids:
            for p, c, t in view_first[pid]:
                query.append((p, pid, c, t))
            for p, c, t in view_second[pid]:
                gallery.append((p, pid, c, t))
        self._finalize([], query, gallery,
                       f"CUHK01 (unseen, {eval_direction}, split {base_split})", verbose)


class CUHK02(BaseImageDataset):
    """``<root>/cuhk02/Dataset/P5/{cam1,cam2}/*.png`` (P5 is the test split)."""

    dataset_dir = "cuhk02"

    def __init__(self, root: str = "", verbose: bool = True,
                 split_id: int = 0, seed: int = 123, direction: str = "A2B", **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.p5_cam1 = osp.join(self.dataset_dir, "Dataset", "P5", "cam1")
        self.p5_cam2 = osp.join(self.dataset_dir, "Dataset", "P5", "cam2")
        for d in (self.dataset_dir, self.p5_cam1, self.p5_cam2):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        cam1 = self._group(self.p5_cam1, camid=0)
        cam2 = self._group(self.p5_cam2, camid=1)
        overlap = sorted(set(cam1) & set(cam2))
        if not overlap:
            raise RuntimeError("CUHK02: no identity overlap between cam1 and cam2 on P5")
        direction = direction if direction in ("A2B", "B2A") else "A2B"
        rng = random.Random(seed + split_id)

        query, gallery = [], []
        for pid in overlap:
            c1_item = rng.choice(cam1[pid])
            c2_item = rng.choice(cam2[pid])
            first, second = (c1_item, c2_item) if direction == "A2B" else (c2_item, c1_item)
            query.append((first[0], pid, first[1], first[2]))
            gallery.append((second[0], pid, second[1], second[2]))
        self._finalize([], query, gallery, f"CUHK02 (unseen, {direction}, split {split_id})", verbose)

    @staticmethod
    def _parse_pid(fname: str) -> int:
        stem = osp.splitext(fname)[0]
        head = stem.split("_")[0]
        if head.isdigit():
            return int(head)
        digits = ""
        for ch in stem:
            if ch.isdigit():
                digits += ch
            else:
                break
        return int(digits) if digits else -1

    def _group(self, dirpath: str, camid: int):
        by_pid = defaultdict(list)
        for p in sorted(glob.glob(osp.join(dirpath, "*.png"))):
            pid = self._parse_pid(osp.basename(p))
            if pid >= 0:
                by_pid[pid].append((p, camid, 0))
        return by_pid


class GRID(BaseImageDataset):
    """``<root>/grid/{probe, gallery}/*.jpeg`` with ``0000_*`` distractors.

    Uses the dataset's own ``features_and_partitions.mat`` (10 official
    folds, 125 test identities each) rather than a hand-picked split.
    """

    dataset_dir = "grid"

    def __init__(self, root: str = "", verbose: bool = True, split_id: int = 0, **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.probe_dir = osp.join(self.dataset_dir, "probe")
        self.gallery_dir = osp.join(self.dataset_dir, "gallery")
        for d in (self.dataset_dir, self.probe_dir, self.gallery_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        try:
            from scipy.io import loadmat
        except ImportError as e:
            raise RuntimeError("GRID requires scipy to read the official .mat partitions (pip install scipy)") from e

        part_path = self._locate_partition_mat()
        if part_path is None:
            raise RuntimeError(
                f"'features_and_partitions.mat' not found under {self.dataset_dir} "
                "(expected at the root or under 'underground_reid/')"
            )
        train_idx_all = loadmat(part_path)["trainIdxAll"][0]
        if not (0 <= split_id < len(train_idx_all)):
            raise ValueError(f"split_id={split_id} out of range [0, {len(train_idx_all)-1}]")
        train_ids = set(int(x) for x in train_idx_all[split_id][0][0][2][0].tolist())

        probe_all = self._collect(self.probe_dir, camid=0, allow_distractor=False)
        gallery_all = self._collect(self.gallery_dir, camid=1, allow_distractor=True)

        query = [(p, pid, c, t) for p, pid, c, t, idx, is_dist in probe_all
                 if idx is not None and idx not in train_ids]
        gallery = [(p, pid, c, t) for p, pid, c, t, idx, is_dist in gallery_all
                   if is_dist or (idx is not None and idx not in train_ids)]
        self._finalize([], query, gallery, f"GRID (unseen, split {split_id})", verbose)

    def _locate_partition_mat(self):
        for candidate in (
            osp.join(self.dataset_dir, "features_and_partitions.mat"),
            osp.join(self.dataset_dir, "underground_reid", "features_and_partitions.mat"),
        ):
            if osp.exists(candidate):
                return candidate
        return None

    @staticmethod
    def _parse_pid(fname: str, allow_distractor: bool) -> int:
        if len(fname) < 4 or not fname[:4].isdigit():
            return -1
        pid = int(fname[:4])
        if pid == 0:
            return 0 if allow_distractor else -1
        return pid

    @staticmethod
    def _parse_idx(fname: str):
        base = osp.splitext(fname)[0]
        if len(base) < 4 or not base[:4].isdigit():
            return None
        idx = int(base[:4])
        return idx if idx != 0 else None

    def _collect(self, dirpath: str, camid: int, allow_distractor: bool):
        paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            paths.extend(glob.glob(osp.join(dirpath, ext)))
        paths = sorted(paths)
        if not paths:
            raise RuntimeError(f"no images under {dirpath}")

        items = []
        for p in paths:
            fname = osp.basename(p)
            pid = self._parse_pid(fname, allow_distractor)
            if pid < 0:
                continue
            items.append((p, pid, camid, 0, self._parse_idx(fname), pid == 0))
        return items


class ILIDS(BaseImageDataset):
    """``<root>/ilids/i-LIDS_Pedestrian/Persons/*.jpg``, filenames ``PPPPSSS.jpg``.

    Following the standard protocol (see e.g. torchreid's ``iLIDS``
    loader), identities are randomly split in half; only the held-out
    test half (~60 of 119) is evaluated, sampling one query and one
    gallery image per identity (single-shot).
    """

    dataset_dir = "ilids"

    def __init__(self, root: str = "", verbose: bool = True,
                 split_id: int = 0, seed: int = 123, **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.persons_dir = osp.join(self.dataset_dir, "i-LIDS_Pedestrian", "Persons")
        for d in (self.dataset_dir, self.persons_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            paths.extend(glob.glob(osp.join(self.persons_dir, ext)))
        paths = sorted(paths)
        if not paths:
            raise RuntimeError(f"no images under {self.persons_dir}")

        by_pid = defaultdict(list)
        for p in paths:
            fname = osp.basename(p)
            if len(fname) < 7:
                continue
            try:
                pid = int(fname[:4])
            except Exception:
                continue
            by_pid[pid].append(p)

        all_pids = sorted(by_pid)
        rng = random.Random(seed + split_id)
        shuffled = all_pids[:]
        rng.shuffle(shuffled)
        num_train = int(len(shuffled) * 0.5)
        test_pids = shuffled[num_train:]

        query, gallery = [], []
        for pid in test_pids:
            imgs = by_pid[pid]
            if len(imgs) < 2:
                continue
            q_path, g_path = rng.sample(imgs, 2)
            query.append((q_path, pid, 0, 0))
            gallery.append((g_path, pid, 1, 0))
        self._finalize([], query, gallery, f"iLIDS (unseen, split {split_id})", verbose)


class PRID(BaseImageDataset):
    """``<root>/prid/single_shot/{cam_a, cam_b}/person_XXXX.png``.

    Standard PRID2011 single-shot protocol: 200 identities appear in
    both cameras; each split randomly holds out 100 as test (the other
    100 are unused). Distractors from the non-overlapping identities
    are kept in the gallery.
    """

    dataset_dir = "prid"

    def __init__(self, root: str = "", verbose: bool = True,
                 split_id: int = 0, seed: int = 123, direction: str = "A2B", **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.cam_a_dir = osp.join(self.dataset_dir, "single_shot", "cam_a")
        self.cam_b_dir = osp.join(self.dataset_dir, "single_shot", "cam_b")
        for d in (self.dataset_dir, self.cam_a_dir, self.cam_b_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        a_map = self._build_map(self.cam_a_dir)
        b_map = self._build_map(self.cam_b_dir)
        overlap = sorted(set(a_map) & set(b_map))
        if len(overlap) < 2:
            raise RuntimeError("PRID: not enough overlapping identities between cam_a and cam_b")
        direction = direction if direction in ("A2B", "B2A") else "A2B"

        pool = overlap[:200] if len(overlap) >= 200 else overlap
        rng = random.Random(seed + split_id)
        shuffled = pool[:]
        rng.shuffle(shuffled)
        half = 100 if len(shuffled) >= 200 else max(1, len(shuffled) // 2)
        test_ids = shuffled[half:half * 2] or shuffled[half:half + 1]

        query, gallery = [], []
        if direction == "A2B":
            for pid in sorted(test_ids):
                query.append((a_map[pid], pid, 0, 0))
                gallery.append((b_map[pid], pid, 1, 0))
            for pid in sorted(set(b_map) - set(overlap)):
                gallery.append((b_map[pid], pid, 1, 0))
        else:
            for pid in sorted(test_ids):
                query.append((b_map[pid], pid, 1, 0))
                gallery.append((a_map[pid], pid, 0, 0))
            for pid in sorted(set(a_map) - set(overlap)):
                gallery.append((a_map[pid], pid, 0, 0))
        self._finalize([], query, gallery, f"PRID (unseen, {direction}, split {split_id})", verbose)

    @staticmethod
    def _parse_pid(fname: str):
        if "person_" not in fname:
            return None
        stem = fname.split("person_", 1)[1]
        digits = ""
        for ch in stem:
            if ch.isdigit():
                digits += ch
            else:
                break
        return int(digits) if digits else None

    def _build_map(self, dirpath: str):
        paths = []
        for ext in ("*.png", "*.jpg", "*.jpeg"):
            paths.extend(glob.glob(osp.join(dirpath, ext)))
        m = {}
        for p in sorted(paths):
            pid = self._parse_pid(osp.basename(p))
            if pid is not None:
                m.setdefault(pid, p)
        if not m:
            raise RuntimeError(f"no valid images in {dirpath}")
        return m


class SenseReID(BaseImageDataset):
    """``<root>/sensereid/SenseReID/{test_probe, test_gallery}/<pid>_<camid>.jpg``.

    Query/gallery are fixed by the dataset's own release.
    """

    dataset_dir = "sensereid"

    def __init__(self, root: str = "", verbose: bool = True, **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.query_dir = osp.join(self.dataset_dir, "SenseReID", "test_probe")
        self.gallery_dir = osp.join(self.dataset_dir, "SenseReID", "test_gallery")
        for d in (self.dataset_dir, self.query_dir, self.gallery_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        query = self._collect(self.query_dir)
        gallery = self._collect(self.gallery_dir)
        self._finalize([], query, gallery, "SenseReID (unseen)", verbose)

    @staticmethod
    def _parse(fname: str):
        stem = osp.splitext(fname)[0]
        if "_" not in stem:
            return None, None
        a, b = stem.split("_", 1)
        if not a.isdigit() or not b.isdigit():
            return None, None
        return int(a), int(b)

    def _collect(self, dirpath: str):
        paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            paths.extend(glob.glob(osp.join(dirpath, ext)))
        paths = sorted(paths)
        if not paths:
            raise RuntimeError(f"no images under {dirpath}")
        items = []
        for p in paths:
            pid, camid = self._parse(osp.basename(p))
            if pid is not None:
                items.append((p, pid, camid, 0))
        return items


class VIPeR(BaseImageDataset):
    """``<root>/viper/VIPeR/{cam_a, cam_b}/*.bmp``; i-th images share the same identity.

    Standard VIPeR generalisation protocol: a random 50/50 identity
    split (test half only), evaluated in both directions and averaged
    over 10 random splits (20 runs total).
    """

    dataset_dir = "viper"

    def __init__(self, root: str = "", verbose: bool = True,
                 split_id: int = 0, seed: int = 123, direction: str = "auto", **kwargs):
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.cam_a_dir = self._resolve(osp.join(self.dataset_dir, "VIPeR", "cam_a"))
        self.cam_b_dir = self._resolve(osp.join(self.dataset_dir, "VIPeR", "cam_b"))
        for d in (self.dataset_dir, self.cam_a_dir, self.cam_b_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' is not available")

        a_paths = self._glob(self.cam_a_dir)
        b_paths = self._glob(self.cam_b_dir)
        if not a_paths or not b_paths or len(a_paths) != len(b_paths):
            raise RuntimeError(f"VIPeR: cam_a={len(a_paths)} cam_b={len(b_paths)} (must be equal and nonempty)")
        N = len(a_paths)

        if direction in ("A2B", "B2A"):
            eval_direction, base_split = direction, split_id
        else:
            eval_direction = "A2B" if split_id % 2 == 0 else "B2A"
            base_split = split_id // 2

        rng = random.Random(seed + base_split)
        shuffled = list(range(1, N + 1))
        rng.shuffle(shuffled)
        test_ids = sorted(shuffled[len(shuffled) // 2:])

        def a_of(pid):
            return a_paths[pid - 1]

        def b_of(pid):
            return b_paths[pid - 1]

        if eval_direction == "A2B":
            query = [(a_of(pid), pid, 0, 0) for pid in test_ids]
            gallery = [(b_of(pid), pid, 1, 0) for pid in test_ids]
        else:
            query = [(b_of(pid), pid, 1, 0) for pid in test_ids]
            gallery = [(a_of(pid), pid, 0, 0) for pid in test_ids]
        self._finalize([], query, gallery,
                       f"VIPeR (unseen, {eval_direction}, split {base_split})", verbose)

    @staticmethod
    def _resolve(base: str) -> str:
        maybe = osp.join(base, "images")
        return maybe if osp.isdir(maybe) else base

    @staticmethod
    def _glob(dirpath: str):
        paths = []
        for ext in ("*.bmp", "*.BMP", "*.jpg", "*.jpeg", "*.png"):
            paths.extend(glob.glob(osp.join(dirpath, ext)))
        return sorted(paths)


class RandomIdentitySampler(Sampler):
    """Sample ``num_instances`` images per identity to form each mini-batch."""

    def __init__(self, data_source, batch_size: int, num_instances: int):
        if batch_size < num_instances:
            raise ValueError("batch_size must be >= num_instances")
        self.data_source = data_source
        self.batch_size = batch_size
        self.num_instances = num_instances
        self.num_pids_per_batch = self.batch_size // self.num_instances

        self.index_dic = defaultdict(list)
        for index, (_, pid, _, _) in enumerate(self.data_source):
            self.index_dic[pid].append(index)
        self.pids = list(self.index_dic.keys())

        self.length = 0
        for pid in self.pids:
            n = max(len(self.index_dic[pid]), self.num_instances)
            self.length += n - n % self.num_instances

    def __iter__(self):
        batch_idxs_dict = defaultdict(list)
        for pid in self.pids:
            idxs = copy.deepcopy(self.index_dic[pid])
            if len(idxs) < self.num_instances:
                idxs = np.random.choice(idxs, size=self.num_instances, replace=True).tolist()
            random.shuffle(idxs)
            batch_idxs = []
            for idx in idxs:
                batch_idxs.append(idx)
                if len(batch_idxs) == self.num_instances:
                    batch_idxs_dict[pid].append(batch_idxs)
                    batch_idxs = []

        avai_pids = copy.deepcopy(self.pids)
        final_idxs = []
        while len(avai_pids) >= self.num_pids_per_batch:
            selected = random.sample(avai_pids, self.num_pids_per_batch)
            for pid in selected:
                final_idxs.extend(batch_idxs_dict[pid].pop(0))
                if not batch_idxs_dict[pid]:
                    avai_pids.remove(pid)
        self.length = len(final_idxs)
        return iter(final_idxs)

    def __len__(self):
        return self.length


class GenericMarketStyle(BaseImageDataset):
    """Fallback (not in the official repo) for uploads that ship MSMT17/CUHK03 already in Market format,
    e.g. `0001_c2_0001.png`. Same parsing rules as the official Market/CUHK03 loaders, accepts jpg+png."""

    def __init__(self, root: str = "", name: str = "", verbose: bool = True, **kwargs):
        d = osp.join(root, name)
        self.pid_begin = 0
        train = self._process_dir(osp.join(d, "bounding_box_train"), relabel=True)
        query = self._process_dir(osp.join(d, "query"), relabel=False)
        gallery = self._process_dir(osp.join(d, "bounding_box_test"), relabel=False)
        self._finalize(train, query, gallery, f"{name} (market-style)", verbose)

    def _process_dir(self, dir_path, relabel=False):
        img_paths = sorted(glob.glob(osp.join(dir_path, "*.jpg")) + glob.glob(osp.join(dir_path, "*.png")))
        pattern = re.compile(r"([-\d]+)_c(\d+)")
        parsed = [(p, pattern.search(osp.basename(p))) for p in img_paths]
        parsed = [(p, int(m.group(1)), int(m.group(2))) for p, m in parsed if m is not None and int(m.group(1)) != -1]
        pid2label = {pid: lab for lab, pid in enumerate(sorted({pid for _, pid, _ in parsed}))}
        return [(p, pid2label[pid] if relabel else pid, cam - 1, 0) for p, pid, cam in parsed]


_FACTORY = {
    "market1501": Market1501, "dukemtmcreid": DukeMTMCreID, "msmt17": MSMT17, "cuhksysu": CUHKSYSU,
    "cuhk03": CUHK03, "lpw_s2": LPW_S2,
    "cuhk01": CUHK01, "cuhk02": CUHK02, "grid": GRID, "ilids": ILIDS, "prid": PRID, "sensereid": SenseReID,
    "viper": VIPeR,
}


def build_dataset(name, cfg, verbose=True):
    root = cfg.DATASETS.ROOT_DIR
    kw = dict(split_id=int(cfg.EVAL.SPLIT_ID), seed=int(cfg.EVAL.SEED), direction=str(cfg.EVAL.DIRECTION))
    d = osp.join(root, name)
    if name == "msmt17" and not osp.exists(osp.join(d, "list_train.txt")):
        return GenericMarketStyle(root=root, name=name, verbose=verbose)
    if name == "cuhk03" and not glob.glob(osp.join(d, "bounding_box_train", "*.jpg")):
        return GenericMarketStyle(root=root, name=name, verbose=verbose)
    return _FACTORY[name](root=root, verbose=verbose, **kw)

## DataLoaders (port of `pad/data/loader.py`)
* **Stage-2 loader:** strong augmentation (flip, pad+crop, random erasing) plus the **PK identity sampler**
  (P = batch/4 identities × K = 4 images each). The batch-hard triplet loss needs this.
* **Stage-1 loader:** test-time transform, shuffled. Used only once, to cache frozen image features.
* **Val loader:** query followed by gallery.

In [ ]:
def _train_collate(batch):
    imgs, pids, camids, viewids, _ = zip(*batch)
    return (torch.stack(imgs, dim=0), torch.tensor(pids, dtype=torch.int64),
            torch.tensor(camids, dtype=torch.int64), torch.tensor(viewids, dtype=torch.int64))


def _val_collate(batch):
    imgs, pids, camids, viewids, img_paths = zip(*batch)
    return (torch.stack(imgs, dim=0), pids, camids, torch.tensor(camids, dtype=torch.int64),
            torch.tensor(viewids, dtype=torch.int64), img_paths)


def _quick_subsample_eval(query, gallery, seed=0):
    """Quick mode only: shrink query/gallery so evaluation takes seconds (metrics are NOT comparable)."""
    rng = random.Random(seed)
    q = rng.sample(query, min(len(query), QUICK["eval_max_query"]))
    qp = {x[1] for x in q}
    g_pos = [x for x in gallery if x[1] in qp]
    g_neg = [x for x in gallery if x[1] not in qp]
    g = g_pos + rng.sample(g_neg, max(0, min(len(g_neg), QUICK["eval_max_gallery"] - len(g_pos))))
    return q, g


def make_dataloader(cfg, dataset=None):
    train_tf = T.Compose([
        T.Resize(cfg.INPUT.SIZE_TRAIN, interpolation=InterpolationMode.BICUBIC),   # == interpolation=3
        T.RandomHorizontalFlip(p=cfg.INPUT.PROB),
        T.Pad(cfg.INPUT.PADDING),
        T.RandomCrop(cfg.INPUT.SIZE_TRAIN),
        T.ToTensor(),
        T.Normalize(mean=cfg.INPUT.PIXEL_MEAN, std=cfg.INPUT.PIXEL_STD),
        RandomErasing(probability=cfg.INPUT.RE_PROB, mode="pixel", max_count=1, device="cpu"),
    ])
    val_tf = T.Compose([
        T.Resize(cfg.INPUT.SIZE_TEST),
        T.ToTensor(),
        T.Normalize(mean=cfg.INPUT.PIXEL_MEAN, std=cfg.INPUT.PIXEL_STD),
    ])
    name = cfg.DATASETS.NAMES[0]
    if dataset is None:
        dataset = build_dataset(name, cfg)
    query, gallery = dataset.query, dataset.gallery
    if RUN_MODE == "quick":
        query, gallery = _quick_subsample_eval(query, gallery)

    nw = cfg.DATALOADER.NUM_WORKERS
    num_classes = dataset.num_train_pids
    train_set = ImageDataset(dataset.train, train_tf)
    train_set_normal = ImageDataset(dataset.train, val_tf)
    val_set = ImageDataset(query + gallery, val_tf)

    train_loader_stage2 = train_loader_stage1 = None
    if len(dataset.train) > 0:
        train_loader_stage2 = DataLoader(
            train_set, batch_size=cfg.SOLVER.STAGE2.IMS_PER_BATCH,
            sampler=RandomIdentitySampler(dataset.train, cfg.SOLVER.STAGE2.IMS_PER_BATCH, cfg.DATALOADER.NUM_INSTANCE),
            num_workers=nw, collate_fn=_train_collate, pin_memory=True, persistent_workers=nw > 0)
        train_loader_stage1 = DataLoader(
            train_set_normal, batch_size=cfg.SOLVER.STAGE1.IMS_PER_BATCH, shuffle=True,
            num_workers=nw, collate_fn=_train_collate, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=cfg.TEST.IMS_PER_BATCH, shuffle=False,
                            num_workers=nw, collate_fn=_val_collate, pin_memory=True)
    return train_loader_stage2, train_loader_stage1, val_loader, len(query), num_classes

## CLIP ViT-B/16 backbone (port of `pad/clip/model.py`, ViT part only)
This is the standard OpenAI CLIP ViT with one PAD change in `VisionTransformer.forward`. At **every**
transformer block, the VA-Prompt tokens `[G-Prompt ; top-k E-Prompts]` are inserted right after `[CLS]`, the block
runs, and the prompt tokens are **stripped** again. Patch positions therefore never shift, and each layer gets
fresh prompts. The positional embedding is bilinearly resized from 14×14 to 16×8 patches for 256×128 inputs.

In [ ]:
class LayerNorm(nn.LayerNorm):
    """Subclass torch's LayerNorm to handle fp16."""
    def forward(self, x):
        orig_type = x.dtype
        return super().forward(x.type(torch.float32)).type(orig_type)


class QuickGELU(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(1.702 * x)


class ResidualAttentionBlock(nn.Module):
    def __init__(self, d_model, n_head, attn_mask=None):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_head)
        self.ln_1 = LayerNorm(d_model)
        self.mlp = nn.Sequential(OrderedDict([
            ("c_fc", nn.Linear(d_model, d_model * 4)), ("gelu", QuickGELU()), ("c_proj", nn.Linear(d_model * 4, d_model))]))
        self.ln_2 = LayerNorm(d_model)
        self.attn_mask = attn_mask

    def attention(self, x):
        self.attn_mask = self.attn_mask.to(dtype=x.dtype, device=x.device) if self.attn_mask is not None else None
        return self.attn(x, x, x, need_weights=False, attn_mask=self.attn_mask)[0]

    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class Transformer(nn.Module):
    def __init__(self, width, layers, heads, attn_mask=None):
        super().__init__()
        self.width, self.layers = width, layers
        self.resblocks = nn.Sequential(*[ResidualAttentionBlock(width, heads, attn_mask) for _ in range(layers)])

    def forward(self, x):
        return self.resblocks(x)


class VisionTransformer(nn.Module):
    """CLIP ViT visual encoder with optional per-layer VA-Prompt injection (PAD Sec. 3.4.2)."""

    def __init__(self, h_resolution, w_resolution, patch_size, stride_size, width, layers, heads, output_dim):
        super().__init__()
        self.h_resolution, self.w_resolution, self.output_dim = h_resolution, w_resolution, output_dim
        self.conv1 = nn.Conv2d(3, width, kernel_size=patch_size, stride=stride_size, bias=False)
        scale = width ** -0.5
        self.class_embedding = nn.Parameter(scale * torch.randn(width))
        self.positional_embedding = nn.Parameter(scale * torch.randn(h_resolution * w_resolution + 1, width))
        self.ln_pre = LayerNorm(width)
        self.transformer = Transformer(width, layers, heads)
        self.ln_post = LayerNorm(width)
        self.proj = nn.Parameter(scale * torch.randn(width, output_dim))
        self.va_strip_after_block = True

    def forward(self, x):
        x = self.conv1(x)
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        cls_tok = self.class_embedding.to(x.dtype).expand(x.shape[0], 1, -1)
        x = torch.cat([cls_tok, x], dim=1)
        x = x + self.positional_embedding.to(x.dtype)
        x = self.ln_pre(x).permute(1, 0, 2)                     # NLD -> LND

        use_va = hasattr(self, "va_prompt") and (self.va_prompt is not None)
        idx_pen = min(10, len(self.transformer.resblocks) - 1)  # penultimate block output, used by VISKD
        x_pen_lnd = None
        for li, block in enumerate(self.transformer.resblocks):
            if use_va:
                x_nld = x.permute(1, 0, 2)
                out = self.va_prompt.forward_for_layer(li, x_nld, cls_feat=None)
                prompts, strip_len = out["batched"], out["strip_len"]
                x_cat = torch.cat([x_nld[:, :1, :], prompts, x_nld[:, 1:, :]], dim=1)   # [CLS | prompts | patches]
                x_after = block(x_cat.permute(1, 0, 2)).permute(1, 0, 2)
                if self.va_strip_after_block:
                    x_after = torch.cat([x_after[:, :1, :], x_after[:, 1 + strip_len:, :]], dim=1)
                x = x_after.permute(1, 0, 2)
            else:
                x = block(x)
            if li == idx_pen:
                x_pen_lnd = x.clone()
        x_last_lnd = x
        x_pen = (x_pen_lnd if x_pen_lnd is not None else x_last_lnd).permute(1, 0, 2)
        x_last = self.ln_post(x_last_lnd.permute(1, 0, 2))
        x_proj = x_last @ self.proj if self.proj is not None else x_last
        return x_pen, x_last, x_proj


class CLIP(nn.Module):
    def __init__(self, embed_dim, vision_layers, vision_width, vision_patch_size, vision_stride_size,
                 context_length, vocab_size, transformer_width, transformer_heads, transformer_layers,
                 h_resolution, w_resolution):
        super().__init__()
        self.context_length = context_length
        self.visual = VisionTransformer(h_resolution, w_resolution, vision_patch_size, vision_stride_size,
                                        vision_width, vision_layers, vision_width // 64, embed_dim)
        self.transformer = Transformer(transformer_width, transformer_layers, transformer_heads,
                                       attn_mask=self.build_attention_mask())
        self.vocab_size = vocab_size
        self.token_embedding = nn.Embedding(vocab_size, transformer_width)
        self.positional_embedding = nn.Parameter(torch.empty(self.context_length, transformer_width))
        self.ln_final = LayerNorm(transformer_width)
        self.text_projection = nn.Parameter(torch.empty(transformer_width, embed_dim))
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def build_attention_mask(self):
        mask = torch.empty(self.context_length, self.context_length)
        mask.fill_(float("-inf"))
        mask.triu_(1)
        return mask

    @property
    def dtype(self):
        return self.visual.conv1.weight.dtype


def resize_pos_embed(posemb, posemb_new, hight, width):
    ntok_new = posemb_new.shape[0] - 1
    posemb_token, posemb_grid = posemb[:1], posemb[1:]
    gs_old = int(math.sqrt(len(posemb_grid)))
    print(f"Resized position embedding: {tuple(posemb.shape)} -> {tuple(posemb_new.shape)} (grid {gs_old}x{gs_old} -> {hight}x{width})")
    posemb_grid = posemb_grid.reshape(1, gs_old, gs_old, -1).permute(0, 3, 1, 2)
    posemb_grid = F.interpolate(posemb_grid, size=(hight, width), mode="bilinear")
    posemb_grid = posemb_grid.permute(0, 2, 3, 1).reshape(1, hight * width, -1)
    return torch.cat([posemb_token, posemb_grid.squeeze()], dim=0)


def convert_weights(model):
    """(Official name kept) -- actually casts Linear/Conv/MHA/projection weights to fp32."""
    def _f(l):
        if isinstance(l, (nn.Conv1d, nn.Conv2d, nn.Linear)):
            l.weight.data = l.weight.data.float()
            if l.bias is not None:
                l.bias.data = l.bias.data.float()
        if isinstance(l, nn.MultiheadAttention):
            for attr in [*[f"{s}_proj_weight" for s in ["in", "q", "k", "v"]], "in_proj_bias", "bias_k", "bias_v"]:
                t = getattr(l, attr)
                if t is not None:
                    t.data = t.data.float()
        for name in ["text_projection", "proj"]:
            if hasattr(l, name):
                a = getattr(l, name)
                if a is not None:
                    a.data = a.data.float()
    model.apply(_f)


def build_model(state_dict, h_resolution, w_resolution, vision_stride_size):
    state_dict = {k: v for k, v in state_dict.items() if k not in ("input_resolution", "context_length", "vocab_size")}
    vision_width = state_dict["visual.conv1.weight"].shape[0]
    vision_layers = len([k for k in state_dict if k.startswith("visual.") and k.endswith(".attn.in_proj_weight")])
    vision_patch_size = state_dict["visual.conv1.weight"].shape[-1]
    embed_dim = state_dict["text_projection"].shape[1]
    context_length = state_dict["positional_embedding"].shape[0]
    vocab_size = state_dict["token_embedding.weight"].shape[0]
    transformer_width = state_dict["ln_final.weight"].shape[0]
    transformer_layers = len(set(k.split(".")[2] for k in state_dict if k.startswith("transformer.resblocks")))
    model = CLIP(embed_dim, vision_layers, vision_width, vision_patch_size, vision_stride_size, context_length,
                 vocab_size, transformer_width, transformer_width // 64, transformer_layers, h_resolution, w_resolution)
    pe = state_dict["visual.positional_embedding"]
    try:   # official code interpolates in the checkpoint dtype (fp16 for the OpenAI JIT file)
        state_dict["visual.positional_embedding"] = resize_pos_embed(pe, model.visual.positional_embedding, h_resolution, w_resolution)
    except RuntimeError:   # older CPU kernels lack fp16 bilinear
        state_dict["visual.positional_embedding"] = resize_pos_embed(pe.float(), model.visual.positional_embedding, h_resolution, w_resolution)
    convert_weights(model)
    expected = set(model.state_dict().keys())
    state_dict = {k: v for k, v in state_dict.items() if k in expected}   # drop extras some mirrors carry
    model.load_state_dict(state_dict, strict=True)
    return model.eval()


# ---- CLIP weights: tried in order; the first that works is used -----------------------------------------------------
CLIP_SHA = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"
CLIP_URLS = [f"https://openaipublic.azureedge.net/clip/models/{CLIP_SHA}/ViT-B-16.pt",          # official (as in repo)
             f"https://openaipublic.blob.core.windows.net/clip/models/{CLIP_SHA}/ViT-B-16.pt"]  # same file, origin host


def _clip_state_dict_from_file(path):
    try:
        return torch.jit.load(path, map_location="cpu").eval().state_dict()
    except RuntimeError:
        sd = torch.load(path, map_location="cpu", weights_only=False)
        return sd.get("state_dict", sd) if isinstance(sd, dict) else sd.state_dict()


_CLIP_SD_CACHE = {}


def load_clip_state_dict():
    if "sd" in _CLIP_SD_CACHE:
        return {k: v.clone() for k, v in _CLIP_SD_CACHE["sd"].items()}
    sd = None
    # 1) a copy attached as a Kaggle input (any file named ViT-B-16.pt), or cached from an earlier cell run
    local = [osp.join(CLIP_CACHE, "ViT-B-16.pt")] + [osp.join(d, "ViT-B-16.pt") for d in ALL_INPUT_DIRS]
    for p in local:
        if osp.isfile(p):
            sd = _clip_state_dict_from_file(p); print(f"[CLIP] loaded {p}"); break
    # 2) the official OpenAI URL used by the repo (sha256-verified)
    if sd is None:
        target = osp.join(CLIP_CACHE, "ViT-B-16.pt")
        for url in CLIP_URLS:
            try:
                print(f"[CLIP] downloading {url}")
                urllib.request.urlretrieve(url, target + ".part")
                if hashlib.sha256(open(target + ".part", "rb").read()).hexdigest() != CLIP_SHA:
                    raise RuntimeError("sha256 mismatch")
                os.replace(target + ".part", target)
                sd = _clip_state_dict_from_file(target); break
            except Exception as e:  # noqa: BLE001
                print(f"[CLIP]   failed: {e}")
    # 3) the same OpenAI weights mirrored on the Hugging Face hub (open_clip format, identical key names)
    if sd is None:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download("timm/vit_base_patch16_clip_224.openai", "open_clip_pytorch_model.bin", cache_dir=CLIP_CACHE)
        sd = torch.load(p, map_location="cpu", weights_only=False); print(f"[CLIP] loaded HF mirror {p}")
    _CLIP_SD_CACHE["sd"] = {k: v.clone() for k, v in sd.items()}
    return sd

## PAD model (port of `pad/model.py`)
* **`VAPromptPool`** (visual side): for each of the 12 layers there is one shared **G-Prompt** (6 tokens) and a pool
  of 36 **E-Prompt** slots (6 tokens each) with keys. A query (L2-normalised mean token of the layer input) picks
  the **top-4** slots by cosine similarity. Each new domain opens fresh slots (8, or 4 for CUHK03), and slots of
  earlier domains are **frozen** with a gradient mask.
* **`TAPromptLearner`** (text side): `"A photo of a [X1][X2][X3][X4] person."`, where the 4 `[X]` tokens are
  learnable **per identity**. The CLIP text encoder stays frozen.
* **`PADModel`**: two BNNeck heads (768-d CLS and 512-d projected CLS), ID classifiers, and the text branch.

The prompt template is tokenised once **offline**, so the notebook needs neither the CLIP BPE vocabulary file nor
`ftfy`/`regex`. The IDs below are exactly what `clip.tokenize("A photo of a X X X X person.")` returns.

In [ ]:
# <|startoftext|> a photo of a x x x x person . <|endoftext|>  (padded with zeros to 77)
_PROMPT_TOKEN_IDS = [49406, 320, 1125, 539, 320, 343, 343, 343, 343, 2533, 269, 49407]


def tokenize_prompt_template(context_length=77):
    t = torch.zeros(1, context_length, dtype=torch.long)
    t[0, :len(_PROMPT_TOKEN_IDS)] = torch.tensor(_PROMPT_TOKEN_IDS)
    return t


def _l2(x, dim=-1, eps=1e-12):
    return x / x.norm(p=2, dim=dim, keepdim=True).clamp(min=eps)


class VAPromptPool(nn.Module):
    """Per-layer {G-Prompt, E-Prompt pool} with lifelong slot allocation."""

    def __init__(self, num_layers, embed_dim=768, g_len=6, e_len=6, pool_size=36, top_k=4,
                 embedding_key="mean", key_learnable=True):
        super().__init__()
        assert num_layers > 0 and 0 < top_k <= pool_size and embedding_key in ("mean", "cls")
        self.num_layers, self.embed_dim, self.g_len, self.e_len = num_layers, embed_dim, g_len, e_len
        self.pool_size, self.top_k, self.embedding_key, self.key_learnable = pool_size, top_k, embedding_key, key_learnable
        self.g_prompt = nn.ParameterList(nn.Parameter(self._init((g_len, embed_dim))) for _ in range(num_layers))
        self.e_prompt = nn.ParameterList(nn.Parameter(self._init((pool_size, e_len, embed_dim))) for _ in range(num_layers))
        if key_learnable:
            self.e_key = nn.ParameterList(nn.Parameter(self._init((pool_size, embed_dim))) for _ in range(num_layers))
        else:
            self.e_key = nn.ParameterList(nn.Parameter(torch.empty(0), requires_grad=False) for _ in range(num_layers))
        self.register_buffer("active_sizes", torch.zeros(num_layers, 1, dtype=torch.long))
        self.register_buffer("frozen_masks", torch.zeros(num_layers, pool_size, dtype=torch.bool))
        self.register_buffer("trainable_masks", torch.zeros(num_layers, pool_size, dtype=torch.bool))

    @staticmethod
    def _init(shape):
        t = torch.zeros(shape)
        nn.init.uniform_(t, a=-0.02, b=0.02)
        return t

    def _query(self, x_noprompt, cls_feat):
        q = x_noprompt.mean(dim=1) if self.embedding_key == "mean" else (cls_feat if cls_feat is not None else x_noprompt[:, 0, :])
        return _l2(q, dim=-1)

    def _keys(self, layer_id):
        k = self.e_key[layer_id] if self.key_learnable else self.e_prompt[layer_id].mean(dim=1)
        return _l2(k, dim=-1)

    @torch.no_grad()
    def _topk(self, layer_id, q):
        keys = self._keys(layer_id)
        used = int(self.active_sizes[layer_id, 0].item())
        if used <= 0:
            used = min(self.pool_size, self.top_k)
        sim = q @ keys[:used].t()
        _, idx = torch.topk(sim, k=min(self.top_k, used), dim=1)
        return idx

    def forward_for_layer(self, layer_id, x_noprompt, cls_feat=None):
        B, _, C = x_noprompt.shape
        q = self._query(x_noprompt, cls_feat)
        idx = self._topk(layer_id, q)
        ep = self.e_prompt[layer_id][idx].reshape(B, -1, C).to(device=x_noprompt.device, dtype=x_noprompt.dtype)
        gp = self.g_prompt[layer_id].unsqueeze(0).expand(B, -1, -1).to(device=x_noprompt.device, dtype=x_noprompt.dtype)
        batched = torch.cat([gp, ep], dim=1)
        return {"batched": batched, "strip_len": batched.size(1), "topk_idx": idx}

    @torch.no_grad()
    def allocate_new_domain_slots(self, slots_per_layer):
        """Freeze previously-used slots and open `slots_per_layer` fresh ones."""
        for li in range(self.num_layers):
            used_before = int(self.active_sizes[li, 0].item())
            used_after = min(used_before + slots_per_layer, self.pool_size)
            frozen = self.frozen_masks[li].clone()
            if used_before > 0:
                frozen[:used_before] = True
            self.frozen_masks[li] = frozen
            trainable = torch.zeros_like(self.trainable_masks[li])
            if used_after > used_before:
                trainable[used_before:used_after] = True
            self.trainable_masks[li] = trainable
            self.active_sizes[li, 0] = used_after


class TextEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.dtype = clip_model.dtype

    def forward(self, prompts, tokenized_prompts):
        x = prompts + self.positional_embedding.type(self.dtype)
        x = self.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)
        x = self.ln_final(x).type(self.dtype)
        eot = tokenized_prompts.argmax(dim=-1)
        return x[torch.arange(x.shape[0]), eot] @ self.text_projection


class TAPromptLearner(nn.Module):
    """Class-conditional "A photo of a X X X X person." tokens (PAD Sec. 3.3)."""

    def __init__(self, num_class, dtype, token_embedding, n_ctx=4, n_cls_ctx=4):
        super().__init__()
        assert n_ctx == 4 and n_cls_ctx == 4, "pre-tokenised template assumes the official N_CTX = N_CLS_CTX = 4"
        ctx_dim = 512
        tokenized = tokenize_prompt_template().to(token_embedding.weight.device)
        with torch.no_grad():
            embedding = token_embedding(tokenized).type(dtype)
        self.tokenized_prompts = tokenized
        cls_vectors = torch.empty(num_class, n_cls_ctx, ctx_dim, dtype=dtype)
        nn.init.normal_(cls_vectors, std=0.02)
        self.cls_ctx = nn.Parameter(cls_vectors)
        self.register_buffer("token_prefix", embedding[:, : n_ctx + 1, :])           # SOT a photo of a
        self.register_buffer("token_suffix", embedding[:, n_ctx + 1 + n_cls_ctx:, :])  # person . EOT ...
        self.num_class, self.n_cls_ctx = num_class, n_cls_ctx

    def forward(self, label):
        b = label.shape[0]
        return torch.cat([self.token_prefix.expand(b, -1, -1), self.cls_ctx[label], self.token_suffix.expand(b, -1, -1)], dim=1)


def _init_kaiming(m):
    cls = m.__class__.__name__
    if cls.find("Linear") != -1:
        nn.init.kaiming_normal_(m.weight, a=0, mode="fan_out"); nn.init.constant_(m.bias, 0.0)
    elif cls.find("Conv") != -1:
        nn.init.kaiming_normal_(m.weight, a=0, mode="fan_in")
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)
    elif cls.find("BatchNorm") != -1 and m.affine:
        nn.init.constant_(m.weight, 1.0); nn.init.constant_(m.bias, 0.0)


def _init_classifier(m):
    if m.__class__.__name__.find("Linear") != -1:
        nn.init.normal_(m.weight, std=0.001)
        if m.bias:
            nn.init.constant_(m.bias, 0.0)


class PADModel(nn.Module):
    def __init__(self, num_classes, cfg):
        super().__init__()
        self.model_name, self.num_classes, self.neck_feat = cfg.MODEL.NAME, num_classes, cfg.TEST.NECK_FEAT
        self.in_planes, self.in_planes_proj = 768, 512
        self.classifier = nn.Linear(self.in_planes, num_classes, bias=False); self.classifier.apply(_init_classifier)
        self.classifier_proj = nn.Linear(self.in_planes_proj, num_classes, bias=False); self.classifier_proj.apply(_init_classifier)
        self.bottleneck = nn.BatchNorm1d(self.in_planes); self.bottleneck.bias.requires_grad_(False); self.bottleneck.apply(_init_kaiming)
        self.bottleneck_proj = nn.BatchNorm1d(self.in_planes_proj); self.bottleneck_proj.bias.requires_grad_(False); self.bottleneck_proj.apply(_init_kaiming)

        h_res = (cfg.INPUT.SIZE_TRAIN[0] - 16) // cfg.MODEL.STRIDE_SIZE[0] + 1
        w_res = (cfg.INPUT.SIZE_TRAIN[1] - 16) // cfg.MODEL.STRIDE_SIZE[1] + 1
        clip_model = build_model(load_clip_state_dict(), h_res, w_res, cfg.MODEL.STRIDE_SIZE[0]).to(DEVICE)
        self.image_encoder = clip_model.visual
        if cfg.VA_PROMPT.ENABLE:
            self.image_encoder.va_prompt = VAPromptPool(
                num_layers=int(self.image_encoder.transformer.layers), embed_dim=self.in_planes,
                g_len=int(cfg.VA_PROMPT.G_LEN), e_len=int(cfg.VA_PROMPT.E_LEN), pool_size=int(cfg.VA_PROMPT.POOL_SIZE),
                top_k=int(cfg.VA_PROMPT.TOP_K), embedding_key=str(cfg.VA_PROMPT.EMBEDDING_KEY),
                key_learnable=bool(cfg.VA_PROMPT.KEY_LEARNABLE)).to(DEVICE)
            self.image_encoder.va_strip_after_block = bool(cfg.VA_PROMPT.STRIP_AFTER_BLOCK)
        self.prompt_learner = TAPromptLearner(num_classes, clip_model.dtype, clip_model.token_embedding,
                                              n_ctx=int(cfg.TA_PROMPT.N_CTX), n_cls_ctx=int(cfg.TA_PROMPT.N_CLS_CTX))
        self.text_encoder = TextEncoder(clip_model)

    def forward(self, x=None, label=None, get_image=False, get_text=False, return_train_outputs=False):
        if get_text:
            return self.text_encoder(self.prompt_learner(label), self.prompt_learner.tokenized_prompts)
        if get_image:
            _, _, proj = self.image_encoder(x)
            return proj[:, 0]
        feat_pen, feat_last, feat_proj = self.image_encoder(x)
        f_pen, f, f_proj = feat_pen[:, 0], feat_last[:, 0], feat_proj[:, 0]
        feat, feat_p = self.bottleneck(f), self.bottleneck_proj(f_proj)
        if self.training or return_train_outputs:
            return [self.classifier(feat), self.classifier_proj(feat_p)], [f_pen, f, f_proj], f_proj
        if self.neck_feat == "after":
            return torch.cat([feat, feat_p], dim=1)
        return torch.cat([f, f_proj], dim=1)          # test feature: [CLS(768) ; proj CLS(512)] before BN

## Losses (verbatim port of `pad/losses.py`)
* `CrossEntropyLabelSmooth`: ID loss, ε = 0.1
* `TripletLoss`: batch-hard triplet, margin 0.3
* `SupConLoss`: image↔text supervised contrastive loss (Stage 1, CLIP-ReID style)
* `KDLosses`: VISKD = λ_feat·MSE(features) + λ_logit·T²·KL(softmax(s/T) ‖ softmax(t/T)). TEXKD reuses `.kl`
* `make_loss`: Stage-2 objective = 0.25·(ID + ID_proj) + 1.0·(Tri_pen + Tri_last + Tri_proj) + 1.0·CE(image→text logits)

In [ ]:
# ---- cross-entropy with label smoothing ----------------------------------
class CrossEntropyLabelSmooth(nn.Module):
    r"""y_smooth = (1 - eps) * y + eps / K."""

    def __init__(self, num_classes: int, epsilon: float = 0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = F.log_softmax(inputs, dim=1)
        with torch.no_grad():
            y = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
            y = (1 - self.epsilon) * y + self.epsilon / self.num_classes
        return (-y * log_probs).mean(0).sum()


# ---- hard-mining triplet loss --------------------------------------------
def _euclidean_dist(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    m, n = x.size(0), y.size(0)
    xx = x.pow(2).sum(1, keepdim=True).expand(m, n)
    yy = y.pow(2).sum(1, keepdim=True).expand(n, m).t()
    dist = xx + yy - 2 * torch.matmul(x, y.t())
    return dist.clamp(min=1e-12).sqrt()


class TripletLoss:
    """Triplet loss with batch-hard mining."""

    def __init__(self, margin: float = None):
        self.margin = margin
        self.ranking_loss = (
            nn.MarginRankingLoss(margin=margin) if margin is not None else nn.SoftMarginLoss()
        )

    def __call__(self, feat: torch.Tensor, labels: torch.Tensor):
        dist_mat = _euclidean_dist(feat, feat)
        N = dist_mat.size(0)
        is_pos = labels.expand(N, N).eq(labels.expand(N, N).t())
        is_neg = labels.expand(N, N).ne(labels.expand(N, N).t())
        dist_ap = dist_mat[is_pos].contiguous().view(N, -1).max(dim=1)[0]
        dist_an = dist_mat[is_neg].contiguous().view(N, -1).min(dim=1)[0]
        y = dist_an.new_ones(dist_an.size())
        if self.margin is not None:
            loss = self.ranking_loss(dist_an, dist_ap, y)
        else:
            loss = self.ranking_loss(dist_an - dist_ap, y)
        return loss, dist_ap, dist_an


# ---- supervised contrastive (used for SupCon image-text alignment) -------
class SupConLoss(nn.Module):
    """Image-to-text supervised contrastive loss (Khosla et al., NeurIPS 2020)."""

    def __init__(self, device: str = "cuda", temperature: float = 1.0):
        super().__init__()
        self.device = device
        self.temperature = temperature

    def forward(self, text_features: torch.Tensor, image_features: torch.Tensor,
                t_label: torch.Tensor, i_label: torch.Tensor) -> torch.Tensor:
        B_t, B_i = text_features.shape[0], image_features.shape[0]
        mask = torch.eq(
            t_label.unsqueeze(1).expand(B_t, B_i),
            i_label.unsqueeze(0).expand(B_t, B_i),
        ).float().to(self.device)

        logits = text_features @ image_features.t() / self.temperature
        logits = logits - logits.max(dim=1, keepdim=True).values.detach()
        log_prob = logits - torch.log(logits.exp().sum(dim=1, keepdim=True))
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        return -mean_log_prob_pos.mean()


# ---- knowledge-distillation (VISKD) --------------------------------------
class KDLosses(nn.Module):
    """Feature-level MSE + logit-level temperature-smoothed KL (PAD Eqs. 7-8)."""

    def __init__(self, lambda_feat: float = 0.5, lambda_logit: float = 0.5,
                 temperature: float = 4.0, feat_layers: Optional[Iterable[int]] = None):
        super().__init__()
        self.lambda_feat = float(lambda_feat)
        self.lambda_logit = float(lambda_logit)
        self.temperature = float(temperature)
        self.feat_layers = list(feat_layers) if feat_layers else None

    def _feat_loss(self, s, t) -> torch.Tensor:
        if not isinstance(s, (list, tuple)):
            s, t = [s], [t]
        idxs = self.feat_layers if self.feat_layers else range(min(len(s), len(t)))
        losses = [
            F.mse_loss(s[i], t[i])
            for i in idxs
            if s[i] is not None and t[i] is not None and s[i].shape == t[i].shape
        ]
        if not losses:
            return torch.zeros((), device=s[0].device)
        return torch.stack(losses).mean()

    def kl(self, s_logits: torch.Tensor, t_logits: torch.Tensor) -> torch.Tensor:
        if s_logits is None or t_logits is None or s_logits.shape != t_logits.shape:
            ref = s_logits if isinstance(s_logits, torch.Tensor) else t_logits
            return torch.zeros((), device=(ref.device if ref is not None else "cpu"))
        T = self.temperature
        log_p_s = F.log_softmax(s_logits / T, dim=1)
        p_t = F.softmax(t_logits.detach() / T, dim=1)
        return F.kl_div(log_p_s, p_t, reduction="batchmean") * (T * T)

    def forward(self, student_feats, teacher_feats, student_logits, teacher_logits):
        device = None
        for ref in (student_logits, teacher_logits):
            if isinstance(ref, torch.Tensor):
                device = ref.device
                break
        if device is None:
            for lst in (student_feats, teacher_feats):
                if isinstance(lst, (list, tuple)) and len(lst):
                    for t in lst:
                        if isinstance(t, torch.Tensor):
                            device = t.device
                            break
                if device is not None:
                    break
        device = device or torch.device("cpu")

        feat_loss = (
            self._feat_loss(student_feats, teacher_feats)
            if self.lambda_feat > 0.0 else torch.zeros((), device=device)
        )
        kl_loss = (
            self.kl(student_logits, teacher_logits)
            if self.lambda_logit > 0.0 else torch.zeros((), device=device)
        )
        total = self.lambda_feat * feat_loss + self.lambda_logit * kl_loss
        return total, feat_loss, kl_loss

    __call__ = forward


# ---- Stage-2 objective builder -------------------------------------------
def make_loss(cfg, num_classes: int):
    """Return a ``(score, feat, target, i2t_score) -> loss`` callable."""
    if cfg.DATALOADER.SAMPLER != "softmax_triplet":
        raise ValueError(f"Unsupported sampler: {cfg.DATALOADER.SAMPLER}")

    triplet = TripletLoss(cfg.SOLVER.MARGIN)
    xent = CrossEntropyLabelSmooth(num_classes) if cfg.MODEL.LABEL_SMOOTH else None

    def _id_loss(score, target):
        if isinstance(score, list):
            return sum(
                xent(s, target) if xent is not None else F.cross_entropy(s, target)
                for s in score
            )
        return xent(score, target) if xent is not None else F.cross_entropy(score, target)

    def _tri_loss(feat, target):
        if isinstance(feat, list):
            return sum(triplet(f, target)[0] for f in feat)
        return triplet(feat, target)[0]

    def loss_fn(score, feat, target, i2t_score=None):
        loss = cfg.MODEL.ID_LOSS_WEIGHT * _id_loss(score, target) \
               + cfg.MODEL.TRIPLET_LOSS_WEIGHT * _tri_loss(feat, target)
        if i2t_score is not None:
            i2t = xent(i2t_score, target) if xent is not None else F.cross_entropy(i2t_score, target)
            loss = loss + cfg.MODEL.I2T_LOSS_WEIGHT * i2t
        return loss

    return loss_fn

## Optimiser and LR schedules (verbatim port of `pad/optim.py`)
Three parameter groups: backbone and heads at `BASE_LR`, and **prompts at `OPT.PROMPT_LR`** (5e-3, or 2e-3 on Market).
This applies in Stage 1 as well, so the TA-Prompt really trains at the prompt LR, not 3.5e-4.
Stage 1 uses timm cosine with 3 warm-up epochs. Stage 2 uses linear warm-up (10 epochs, factor 0.1) and then multistep ×0.1.

In [ ]:
logger = logging.getLogger("pad")


# ---- warmup + multistep (used by Stage 2) --------------------------------
class WarmupMultiStepLR(torch.optim.lr_scheduler._LRScheduler):
    """Linear/constant warm-up followed by multi-step decay. Called per epoch."""

    def __init__(self, optimizer, milestones, gamma: float = 0.1,
                 warmup_factor: float = 1.0 / 3, warmup_iters: int = 500,
                 warmup_method: str = "linear", last_epoch: int = -1):
        if list(milestones) != sorted(milestones):
            raise ValueError(f"milestones must be increasing, got {milestones}")
        if warmup_method not in ("constant", "linear"):
            raise ValueError(f"warmup_method must be constant or linear, got {warmup_method}")
        self.milestones = milestones
        self.gamma = gamma
        self.warmup_factor = warmup_factor
        self.warmup_iters = warmup_iters
        self.warmup_method = warmup_method
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        warmup = 1.0
        if self.last_epoch < self.warmup_iters:
            if self.warmup_method == "constant":
                warmup = self.warmup_factor
            else:
                alpha = self.last_epoch / max(self.warmup_iters, 1)
                warmup = self.warmup_factor * (1 - alpha) + alpha
        decay = self.gamma ** bisect_right(self.milestones, self.last_epoch)
        return [base_lr * warmup * decay for base_lr in self.base_lrs]


def make_cosine_scheduler(optimizer, num_epochs: int, lr_min: float,
                          warmup_lr_init: float, warmup_t: int):
    """Cosine scheduler used by Stage 1 (thin wrapper around timm)."""
    return CosineLRScheduler(
        optimizer,
        t_initial=num_epochs,
        lr_min=lr_min,
        warmup_lr_init=warmup_lr_init,
        warmup_t=warmup_t,
        cycle_limit=1,
        t_in_epochs=True,
    )


# ---- param grouping + optimiser construction -----------------------------
_PROMPT_HINTS: Tuple[str, ...] = (
    "va_prompt", "g_prompt", "e_prompt", "e_key", "prompt_learner",
)
_HEAD_HINTS: Tuple[str, ...] = (
    "classifier", "classifier_proj", "bottleneck", "bottleneck_proj",
)


def _match(name: str, hints: Tuple[str, ...]) -> bool:
    lowered = name.lower()
    return any(h in lowered for h in hints)


def _split_param_groups(model: torch.nn.Module, stage: str):
    prompts, heads, backbones = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        lowered = n.lower()
        if "text_encoder" in lowered:
            continue
        if stage == "STAGE1" and "va_prompt" in lowered:
            continue
        if stage == "STAGE2" and "prompt_learner" in lowered:
            continue
        if _match(lowered, _PROMPT_HINTS):
            prompts.append(p)
        elif _match(lowered, _HEAD_HINTS):
            heads.append(p)
        else:
            backbones.append(p)
    return prompts, heads, backbones


def make_optimizer(cfg, model: torch.nn.Module, stage: str = "STAGE1") -> torch.optim.Optimizer:
    stage_cfg = getattr(cfg.SOLVER, stage)
    base_lr = float(stage_cfg.BASE_LR)
    base_wd = float(getattr(stage_cfg, "WEIGHT_DECAY", 5e-4))
    optim_name = str(getattr(stage_cfg, "OPTIMIZER", "Adam")).lower()

    prompt_lr = float(cfg.OPT.PROMPT_LR) if cfg.OPT.PROMPT_LR > 0 else base_lr
    prompt_wd = float(cfg.OPT.PROMPT_WD)

    prompts, heads, backbones = _split_param_groups(model, stage.upper())

    def _count(ps: List[torch.nn.Parameter]) -> int:
        return sum(p.numel() for p in ps)

    logger.info(
        "[OPT] stage=%s base_lr=%.3g wd=%.3g | prompt_lr=%.3g wd=%.3g | "
        "prompt=%d head=%d backbone=%d",
        stage, base_lr, base_wd, prompt_lr, prompt_wd,
        _count(prompts), _count(heads), _count(backbones),
    )

    groups = []
    if backbones:
        groups.append({"params": backbones, "lr": base_lr, "weight_decay": base_wd, "name": "backbone"})
    if heads:
        groups.append({"params": heads, "lr": base_lr, "weight_decay": base_wd, "name": "head"})
    if prompts:
        groups.append({"params": prompts, "lr": prompt_lr, "weight_decay": prompt_wd, "name": "prompt"})
    if not groups:
        groups.append({
            "params": [p for p in model.parameters() if p.requires_grad],
            "lr": base_lr, "weight_decay": base_wd, "name": "all",
        })

    if optim_name == "sgd":
        return torch.optim.SGD(groups, momentum=0.9, nesterov=True)
    if optim_name == "adamw":
        return torch.optim.AdamW(groups)
    return torch.optim.Adam(groups)

## Utilities: EMA teacher and evaluator (port of `pad/utils.py`)
`EMATeacher` implements θ_tea ← m·θ_tea + (1−m)·θ_stu with m = 0.997, applied to parameters **and** float buffers
(BN statistics) after every optimiser step.

`R1_mAP_eval` computes the same CMC/mAP as the official evaluator: features are L2-normalised, ranked by Euclidean
distance, and gallery images with the same identity **and** the same camera as the query are ignored. The official
version builds the full query×gallery matrix in RAM, which for MSMT17 is 11,659 × 82,161 and needs about 15 GB,
risky on Kaggle. This version processes queries in chunks on the GPU and gives identical numbers.

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = self.avg = self.sum = 0.0
        self.count = 0

    def update(self, val, n=1):
        self.val = val; self.sum += val * n; self.count += n; self.avg = self.sum / max(self.count, 1)


class EMATeacher(nn.Module):
    r"""Momentum teacher: theta_tea = m * theta_tea + (1 - m) * theta_stu  (PAD Eq. 6)."""

    def __init__(self, student, momentum=0.997, device=None):
        super().__init__()
        self.momentum = float(momentum)
        self.teacher = copy.deepcopy(student)
        self.teacher.to(device or next(student.parameters()).device)
        for p in self.teacher.parameters():
            p.requires_grad_(False)
        self.teacher.eval()

    @torch.no_grad()
    def update(self, student, momentum=None):
        m = self.momentum if momentum is None else float(momentum)
        cur_params, cur_buffers = dict(student.named_parameters()), dict(student.named_buffers())
        for name, p in self.teacher.named_parameters():
            if name in cur_params and p.shape == cur_params[name].shape and torch.is_floating_point(p):
                p.mul_(m).add_(cur_params[name].detach().to(device=p.device, dtype=p.dtype), alpha=1.0 - m)
        for name, b in self.teacher.named_buffers():
            if name in cur_buffers and b.shape == cur_buffers[name].shape and torch.is_floating_point(b):
                b.mul_(m).add_(cur_buffers[name].detach().to(device=b.device, dtype=b.dtype), alpha=1.0 - m)

    def forward(self, *args, **kwargs):
        return self.teacher(*args, **kwargs)


@torch.no_grad()
def eval_func_chunked(qf, gf, q_pids, g_pids, q_camids, g_camids, max_rank=50, chunk=256):
    """Same maths as the official `_eval_func` (CMC + mAP with same-id/same-cam removal), computed per query chunk."""
    dev = DEVICE
    gf = gf.to(dev).float()
    g_p, g_c = torch.as_tensor(np.asarray(g_pids), device=dev), torch.as_tensor(np.asarray(g_camids), device=dev)
    q_p, q_c = torch.as_tensor(np.asarray(q_pids), device=dev), torch.as_tensor(np.asarray(q_camids), device=dev)
    num_g = gf.shape[0]
    max_rank = min(max_rank, num_g)
    gg = gf.pow(2).sum(1)
    cmc_sum, aps = torch.zeros(max_rank, device=dev), []
    ranks = torch.arange(max_rank, device=dev)
    for s in range(0, qf.shape[0], chunk):
        q = qf[s:s + chunk].to(dev).float()
        dist = q.pow(2).sum(1, keepdim=True) + gg[None, :] - 2.0 * q @ gf.t()
        order = torch.argsort(dist, dim=1)
        gp, gc = g_p[order], g_c[order]
        qp, qc = q_p[s:s + chunk, None], q_c[s:s + chunk, None]
        match = gp == qp
        keep = ~(match & (gc == qc))
        mk = match & keep
        num_rel = mk.sum(1)
        valid = num_rel > 0
        kept_rank = torch.cumsum(keep.int(), dim=1)                         # 1-based rank among kept gallery items
        first = torch.where(mk, kept_rank, torch.full_like(kept_rank, num_g + 1)).min(1).values
        cmc_sum += ((first[valid, None] - 1) <= ranks[None, :]).float().sum(0)
        prec = torch.where(mk, torch.cumsum(mk.int(), dim=1).float() / kept_rank.clamp(min=1).float(),
                           torch.zeros((), device=dev))
        aps.append((prec.sum(1) / num_rel.clamp(min=1).float())[valid].cpu())
    aps = torch.cat(aps)
    assert aps.numel() > 0, "All query identities are missing from the gallery."
    return (cmc_sum / aps.numel()).cpu().numpy(), float(aps.mean())


class R1_mAP_eval:
    def __init__(self, num_query, max_rank=50, feat_norm="yes"):
        self.num_query, self.max_rank, self.feat_norm = num_query, max_rank, feat_norm
        self.reset()

    def reset(self):
        self.feats, self.pids, self.camids = [], [], []

    def update(self, output):
        feat, pid, camid = output
        self.feats.append(feat.cpu()); self.pids.extend(np.asarray(pid)); self.camids.extend(np.asarray(camid))

    def compute(self):
        feats = torch.cat(self.feats, dim=0)
        if self.feat_norm == "yes":
            feats = F.normalize(feats, dim=1, p=2)
        nq = self.num_query
        return eval_func_chunked(feats[:nq], feats[nq:], self.pids[:nq], self.pids[nq:],
                                 self.camids[:nq], self.camids[nq:], self.max_rank)


def setup_logger(save_dir, fname):
    logger = logging.getLogger("pad")
    logger.setLevel(logging.INFO)
    logger.handlers.clear(); logger.propagate = False
    fmt = logging.Formatter("%(asctime)s %(levelname)s: %(message)s", "%H:%M:%S")
    sh = logging.StreamHandler(sys.stdout); sh.setFormatter(fmt); logger.addHandler(sh)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fh = logging.FileHandler(osp.join(save_dir, fname), mode="a"); fh.setFormatter(fmt); logger.addHandler(fh)
    return logger


def _max_iters():
    return QUICK["max_iters_per_epoch"] if RUN_MODE == "quick" else None


def _autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)

## Training engine (port of `pad/engine.py`)
**Stage 1, TA-Prompt warm-up (paper Sec. 3.3).** The image encoder is frozen, so image features (512-d projected
CLS) are cached once. Only the per-identity text tokens are trained, with symmetric SupCon (image→text +
text→image). From the 2nd domain on, **TEXKD** is added. A frozen copy of the model made *before* Stage 1 gives
fixed text anchors T_e for all classes. For each batch the positive classes plus 256 random negative classes are
taken, and the student's image→text logit distribution is pulled toward the anchor's:
KL(softmax(7·S·T_sᵀ / 0.07) ‖ softmax(7·S·T_eᵀ / 0.07))·0.07², weighted 0.5.

**Stage 2, visual branch (paper Sec. 3.4).** The trained TA-Prompt gives a fixed per-class text bank. The visual
branch trains with ID + triplet + image→text CE. From the 2nd domain on, only the VA-Prompt slots of the current
domain, the last 4 ViT blocks, and the BN/classifier heads are trainable. **VISKD** distils from an **EMA teacher**
(initialised from the previous-domain model): 0.5·MSE over [penultimate CLS, last CLS, projected CLS] plus
0.5·T²·KL over the image→text logits with T = 4.

In [ ]:
def do_train_stage1(cfg, model, train_loader_stage1, optimizer, scheduler, teacher_model=None):
    logger = logging.getLogger("pad")
    logger.info("[Stage1] start")
    epochs, log_period = cfg.SOLVER.STAGE1.MAX_EPOCHS, cfg.SOLVER.STAGE1.LOG_PERIOD
    textkd_on = bool(cfg.TEXTKD.ENABLE) and (teacher_model is not None)
    if textkd_on:
        kd_text = KDLosses(lambda_feat=0.0, lambda_logit=float(cfg.TEXTKD.LAMBDA_LOGIT), temperature=float(cfg.TEXTKD.TEMP))
        logit_scale = float(cfg.TEXTKD.LOGITS_SCALE)
    else:
        kd_text, logit_scale = None, 1.0
    model.to(DEVICE)
    student_ref = model
    total = sum(p.numel() for p in model.parameters()); trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"[Stage1] trainable params: {trainable}/{total} ({trainable / max(total, 1):.2%})")
    loss_meter, scaler, supcon = AverageMeter(), torch.amp.GradScaler("cuda", enabled=USE_AMP), SupConLoss(DEVICE)
    t_start = time.monotonic()

    # Cache image features once (encoder is frozen during Stage 1).
    logger.info("[Stage1] caching image features ...")
    model.eval()
    feats_all, labels_all = [], []
    with torch.no_grad():
        for it, (img, vid, _, _) in enumerate(train_loader_stage1):
            if _max_iters() and it >= _max_iters() * 4:
                break
            img, vid = img.to(DEVICE, non_blocking=True), vid.to(DEVICE, non_blocking=True)
            with _autocast():
                feats_all.append(model(img, vid, get_image=True).detach().float().cpu())
            labels_all.append(vid.detach().cpu())
    feats_all = torch.cat(feats_all).to(DEVICE).float()
    labels_all = torch.cat(labels_all).to(DEVICE)
    N = labels_all.size(0)
    model.train()
    batch = cfg.SOLVER.STAGE1.IMS_PER_BATCH
    iters_per_epoch = (N + batch - 1) // batch
    logger.info(f"[Stage1] cached {N} samples, {iters_per_epoch} iters/epoch")

    # Frozen text-anchor bank for TEXKD.
    Te_all = None
    if textkd_on:
        teacher = teacher_model.to(DEVICE).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        C = int(student_ref.prompt_learner.num_class)
        with torch.no_grad(), _autocast():
            Te_all = teacher(label=torch.arange(C, device=DEVICE, dtype=torch.long), get_text=True).detach().float()
        Te_all = Te_all / Te_all.norm(p=2, dim=1, keepdim=True).clamp_min(1e-6)
    neg_K = int(cfg.TEXTKD.NEG_PER_BATCH)

    for epoch in range(1, epochs + 1):
        loss_meter.reset()
        scheduler.step(epoch)
        model.train()
        kd_running = None
        idx_all = torch.randperm(N, device=DEVICE)
        for it in range(iters_per_epoch):
            s, e = it * batch, min((it + 1) * batch, N)
            if s >= e:
                continue
            idx = idx_all[s:e]
            cur_feats, cur_labels = feats_all[idx], labels_all[idx]
            optimizer.zero_grad(set_to_none=True)
            with _autocast():
                text_feats = model(label=cur_labels, get_text=True).float()
                loss = supcon(cur_feats, text_feats, cur_labels, cur_labels) + supcon(text_feats, cur_feats, cur_labels, cur_labels)
            if textkd_on:
                with torch.no_grad():
                    uniq_pos, _ = torch.unique(cur_labels, sorted=False, return_inverse=True)
                    C = Te_all.size(0)
                    mask = torch.ones(C, dtype=torch.bool, device=Te_all.device); mask[uniq_pos] = False
                    neg_pool = torch.nonzero(mask, as_tuple=False).squeeze(1)
                    if neg_K > 0 and neg_pool.numel() > 0:
                        neg_idx = neg_pool if neg_pool.numel() <= neg_K else neg_pool[torch.randperm(neg_pool.numel(), device=neg_pool.device)[:neg_K]]
                        cls_idx = torch.cat([uniq_pos, neg_idx], dim=0)
                    else:
                        cls_idx = uniq_pos
                with _autocast():
                    Ts_sub = model(label=cls_idx, get_text=True)
                Ts_sub = Ts_sub.float(); Ts_sub = Ts_sub / Ts_sub.norm(p=2, dim=1, keepdim=True).clamp_min(1e-6)
                Te_sub = Te_all.index_select(0, cls_idx)
                S = cur_feats / cur_feats.norm(p=2, dim=1, keepdim=True).clamp_min(1e-6)
                kl = kd_text.kl((S @ Ts_sub.t()) * logit_scale, (S @ Te_sub.t()) * logit_scale)
                loss = loss + kd_text.lambda_logit * kl
                v = float(kl.item()); kd_running = v if kd_running is None else 0.9 * kd_running + 0.1 * v
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            loss_meter.update(loss.item(), cur_feats.size(0))
            if (it + 1) % log_period == 0:
                msg = (f"Stage1 Epoch[{epoch}] Iter[{it + 1}/{iters_per_epoch}] Loss {loss_meter.avg:.3f} "
                       f"LR {optimizer.param_groups[0]['lr']:.2e}")
                if textkd_on and kd_running is not None:
                    msg += f" TEXKD {kd_running:.3f}"
                logger.info(msg)
        logger.info(f"[Stage1] epoch {epoch}/{epochs} loss {loss_meter.avg:.4f}")
    logger.info(f"[Stage1] total time {timedelta(seconds=int(time.monotonic() - t_start))}")


def do_train_stage2(cfg, model, train_loader_stage2, optimizer, scheduler, loss_fn, teacher_model=None):
    logger = logging.getLogger("pad")
    logger.info("[Stage2] start")
    epochs, log_period = cfg.SOLVER.STAGE2.MAX_EPOCHS, cfg.SOLVER.STAGE2.LOG_PERIOD
    model.to(DEVICE)
    student_ref, num_classes = model, model.num_classes
    distill_on = bool(cfg.DISTILL.ENABLE) and (teacher_model is not None)
    ema_teacher = kd_criterion = None
    if distill_on:
        ema_teacher = EMATeacher(teacher_model.to(DEVICE), momentum=float(cfg.DISTILL.EMA_MOMENTUM), device=torch.device(DEVICE))
        kd_criterion = KDLosses(lambda_feat=float(cfg.DISTILL.LAMBDA_FEAT), lambda_logit=float(cfg.DISTILL.LAMBDA_LOGIT),
                                temperature=float(cfg.DISTILL.TEMP))
    loss_meter, acc_meter, scaler = AverageMeter(), AverageMeter(), torch.amp.GradScaler("cuda", enabled=USE_AMP)
    t_start = time.monotonic()

    # Per-class text bank from the trained TA-Prompt (computed once).
    batch = cfg.SOLVER.STAGE2.IMS_PER_BATCH
    text_bank = []
    with torch.no_grad():
        for s in range(0, num_classes, batch):
            with _autocast():
                text_bank.append(model(label=torch.arange(s, min(s + batch, num_classes), device=DEVICE), get_text=True).cpu())
    text_bank = torch.cat(text_bank).to(DEVICE).float()

    for epoch in range(1, epochs + 1):
        t_epoch = time.time()
        loss_meter.reset(); acc_meter.reset()
        scheduler.step()
        model.train()
        n_iter = 0
        for n_iter, (img, vid, _, _) in enumerate(train_loader_stage2):
            if _max_iters() and n_iter >= _max_iters():
                break
            img, target = img.to(DEVICE, non_blocking=True), vid.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with _autocast():
                score, feat, image_features = model(x=img, label=target)
                logits = image_features @ text_bank.t()
                loss = loss_fn(score, feat, target, logits)
                if distill_on:
                    with torch.no_grad():
                        _, t_feat, t_image_features = ema_teacher(x=img, label=target, return_train_outputs=True)
                        t_logits = t_image_features @ text_bank.t()
                    kd_total, kd_feat_val, kd_kl_val = kd_criterion(student_feats=feat, teacher_feats=t_feat,
                                                                    student_logits=logits, teacher_logits=t_logits)
                    loss = loss + kd_total
                else:
                    kd_feat_val = kd_kl_val = None
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            if distill_on:
                ema_teacher.update(student_ref)
            acc = (logits.max(1)[1] == target).float().mean().item()
            loss_meter.update(loss.item(), img.size(0)); acc_meter.update(acc, 1)
            if (n_iter + 1) % log_period == 0:
                msg = (f"Stage2 Epoch[{epoch}] Iter[{n_iter + 1}/{len(train_loader_stage2)}] Loss {loss_meter.avg:.3f} "
                       f"Acc {acc_meter.avg:.3f} LR {optimizer.param_groups[0]['lr']:.2e}")
                if distill_on and kd_feat_val is not None:
                    msg += f" KD_feat {kd_feat_val.item():.3f} KD_kl {kd_kl_val.item():.3f}"
                logger.info(msg)
        dt = (time.time() - t_epoch) / max(n_iter + 1, 1)
        logger.info(f"[Stage2] epoch {epoch}/{epochs} loss {loss_meter.avg:.4f} acc {acc_meter.avg:.3f} "
                    f"({dt:.3f}s/batch, {train_loader_stage2.batch_size / max(dt, 1e-6):.0f} img/s)")
    logger.info(f"[Stage2] total time {timedelta(seconds=int(time.monotonic() - t_start))}")


@torch.no_grad()
def do_inference(cfg, model, val_loader, num_query):
    """fp32 feature extraction, exactly like the official do_inference (no autocast)."""
    evaluator = R1_mAP_eval(num_query, max_rank=50, feat_norm=cfg.TEST.FEAT_NORM)
    model.to(DEVICE).eval()
    for img, pid, camid, _, _, _ in val_loader:
        evaluator.update((model(img.to(DEVICE)), pid, camid))
    cmc, mAP = evaluator.compute()
    return float(cmc[0]), float(cmc[4]), float(mAP)

## Lifelong driver (port of `train_lifelong.py` + `test_lifelong.py` + `scripts/run_lifelong.sh`)
For each domain *t* in order:
1. Build the model from CLIP and load the **visual branch** of the previous step's checkpoint (backbone, VA-Prompt,
   BN necks). The classifier and TA-Prompt are re-created because the identity space changes.
2. `allocate_new_domain_slots`: freeze old E-Prompt slots (gradient mask) and open new ones.
3. Keep `model_old` = a frozen copy, which serves as the TEXKD anchor and the initial EMA teacher.
4. Stage 1 (TA-Prompt), then Stage 2 (visual), then save the checkpoint.
5. Evaluate on every domain seen so far, and append the result to `seq_eval.csv`.

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _set_all_requires_grad(model, flag):
    for p in model.parameters():
        p.requires_grad_(flag)


def _set_by_name(model, substrings, flag):
    keys = tuple(s.lower() for s in ((substrings,) if isinstance(substrings, str) else substrings))
    for n, p in model.named_parameters():
        if any(k in n.lower() for k in keys):
            p.requires_grad_(flag)


def _unfreeze_last_blocks(model, n_last):
    blocks = list(model.image_encoder.transformer.resblocks)
    for i, blk in enumerate(blocks):
        if i >= max(0, len(blocks) - int(n_last)):
            for p in blk.parameters():
                p.requires_grad_(True)


def _log_params(model, logger, tag):
    total = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"[{tag}] trainable params: {tr:,}/{total:,} ({tr / total:.2%})")


def _filter_visual_state_dict(sd):
    """Keep only the visual-branch weights (backbone + VA-Prompt + BN necks) == official _filter_resume_state_dict."""
    out = {}
    for k, v in sd.items():
        while k.startswith("module."):
            k = k[len("module."):]
        lk = k.lower()
        if lk.startswith("image_encoder") or "bottleneck" in lk or \
                (("ln_post" in lk) and ("text" not in lk)) or (lk.endswith(".proj") and "text" not in lk):
            out[k] = v
    return out


def _register_va_grad_mask(model):
    """Zero-out gradients on VA-Prompt slots owned by previous domains."""
    va = getattr(model.image_encoder, "va_prompt", None)
    if va is None:
        return

    def _make_hook(fmask):
        def hook(grad):
            expand = fmask.view(-1, 1, 1) if fmask.ndim == 1 else fmask
            expand = expand.to(grad.device)
            grad = grad.clone()
            grad[expand.expand_as(grad)] = 0
            return grad
        return hook

    for li in range(va.num_layers):
        fmask = va.frozen_masks[li].detach().cpu()
        va.e_prompt[li].register_hook(_make_hook(fmask))                 # [pool, e_len, C]
        if va.key_learnable:
            va.e_key[li].register_hook(_make_hook(fmask.view(-1, 1)))    # [pool, C]


def ckpt_path(domain):
    return osp.join(OUT_DIR, domain, "ViT-B-16_visual.pth")


def find_existing_ckpt(domain):
    """Checkpoint in this session's OUT_DIR, or in an attached previous run's output (Kaggle input)."""
    p = ckpt_path(domain)
    if osp.isfile(p):
        return p
    for d in ALL_INPUT_DIRS:
        q = osp.join(d, "ViT-B-16_visual.pth")
        if osp.basename(d) == domain and "pad_outputs" in d.lower() and osp.isfile(q):
            os.makedirs(osp.dirname(p), exist_ok=True)
            shutil.copy(q, p)
            print(f"[resume] copied {q} -> {p}")
            for f in ("train_log.txt",):
                if osp.isfile(osp.join(d, f)):
                    shutil.copy(osp.join(d, f), osp.join(osp.dirname(p), f))
            return p
    return None


def train_one_domain(domain, prev_ckpt, is_first):
    cfg = build_cfg(domain, is_first)
    set_seed(cfg.SOLVER.SEED)
    out_dir = osp.join(OUT_DIR, domain)
    logger = setup_logger(out_dir, "train_log.txt")
    logger.info(f"[Lifelong] dataset={domain} output_dir={out_dir} RUN_MODE={RUN_MODE}")

    train_loader_stage2, train_loader_stage1, _, _, num_classes = make_dataloader(cfg)
    model = PADModel(num_classes, cfg).to(DEVICE)

    if prev_ckpt:
        filt = _filter_visual_state_dict(torch.load(prev_ckpt, map_location="cpu"))
        ret = model.load_state_dict(filt, strict=False)
        logger.info(f"[Resume] from {prev_ckpt}: loaded={len(filt)} missing={len(ret.missing_keys)} unexpected={len(ret.unexpected_keys)}")
    else:
        logger.info("[Resume] first domain -- starting from CLIP pretrain; KD disabled (no teacher)")
        cfg.DISTILL.ENABLE = False
        cfg.TEXTKD.ENABLE = False
    logger.info(f"[Switches] TEXTKD={cfg.TEXTKD.ENABLE} VISKD={cfg.DISTILL.ENABLE}")

    if hasattr(model.image_encoder, "va_prompt"):
        slots = int(cfg.VA_PROMPT.NEW_SLOTS_PER_DOMAIN)
        model.image_encoder.va_prompt.allocate_new_domain_slots(slots)
        _register_va_grad_mask(model)
        logger.info(f"[VA-Prompt] +{slots} slots/layer; active={int(model.image_encoder.va_prompt.active_sizes[0, 0])}"
                    f"/{cfg.VA_PROMPT.POOL_SIZE}")

    model_old = copy.deepcopy(model).cpu()
    teacher_for_textkd = model_old if (prev_ckpt and cfg.TEXTKD.ENABLE) else None
    loss_func = make_loss(cfg, num_classes=num_classes)

    # ---- Stage 1: TA-Prompt only ----
    _set_all_requires_grad(model, False)
    _set_by_name(model, "prompt_learner", True)
    _log_params(model, logger, "STAGE1")
    optimizer_1 = make_optimizer(cfg, model, stage="STAGE1")
    sched_1 = make_cosine_scheduler(optimizer_1, num_epochs=cfg.SOLVER.STAGE1.MAX_EPOCHS, lr_min=cfg.SOLVER.STAGE1.LR_MIN,
                                    warmup_lr_init=cfg.SOLVER.STAGE1.WARMUP_LR_INIT, warmup_t=cfg.SOLVER.STAGE1.WARMUP_EPOCHS)
    do_train_stage1(cfg, model, train_loader_stage1, optimizer_1, sched_1, teacher_model=teacher_for_textkd)

    # ---- Stage 2: visual branch ----
    if not prev_ckpt:
        _set_all_requires_grad(model, True)
        _set_by_name(model, "prompt_learner", False)
        _log_params(model, logger, "STAGE2 (first domain)")
    else:
        _set_all_requires_grad(model, False)
        _set_by_name(model, "prompt_learner", False)
        _set_by_name(model, ("classifier", "bottleneck"), True)
        if cfg.VA_PROMPT.ENABLE:
            _set_by_name(model, ("va_prompt", "g_prompt", "e_prompt", "e_key"), True)
        _unfreeze_last_blocks(model, int(cfg.UNFREEZE.LAST_BLOCKS))
        _log_params(model, logger, "STAGE2")
    optimizer_2 = make_optimizer(cfg, model, stage="STAGE2")
    sched_2 = WarmupMultiStepLR(optimizer_2, cfg.SOLVER.STAGE2.STEPS, cfg.SOLVER.STAGE2.GAMMA,
                                cfg.SOLVER.STAGE2.WARMUP_FACTOR, cfg.SOLVER.STAGE2.WARMUP_ITERS, cfg.SOLVER.STAGE2.WARMUP_METHOD)
    do_train_stage2(cfg, model, train_loader_stage2, optimizer_2, sched_2, loss_func,
                    teacher_model=(model_old if prev_ckpt else None))

    p = ckpt_path(domain)
    torch.save(_filter_visual_state_dict(model.state_dict()), p)
    logger.info(f"[Save] {p}")
    del model_old, optimizer_1, optimizer_2
    return model, cfg

### Evaluation helper (== `test_lifelong._load_visual_state_dict` + `do_inference`)
The official test script loads the visual weights into a fresh model and then **re-derives the number of active
prompt slots from the non-zero E-Prompt weights**. Because unused slots are randomly initialised (not zero), this
always gives 36, so at test time the top-k is chosen from the whole pool. `OFFICIAL_EVAL_SLOT_RECONSTRUCTION`
reproduces that behaviour (default). Set it to False to evaluate with the slot count used during training.

In [ ]:
def prepare_eval_model(model):
    m = copy.deepcopy(model).to(DEVICE).eval()
    if OFFICIAL_EVAL_SLOT_RECONSTRUCTION and hasattr(m.image_encoder, "va_prompt"):
        va = m.image_encoder.va_prompt
        with torch.no_grad():
            for li in range(va.num_layers):
                used = int((va.e_prompt[li].detach().abs().sum(dim=(1, 2)) > 1e-6).sum().item())
                va.active_sizes[li, 0] = min(used, va.pool_size)
    return m


def model_from_ckpt(ckpt, domain):
    cfg = build_cfg(domain, False)
    m = PADModel(1, cfg).to(DEVICE)
    m.load_state_dict(_filter_visual_state_dict(torch.load(ckpt, map_location="cpu")), strict=False)
    return m


def evaluate_domains(model, eval_domains, trained_domain, csv_out=None, extra_cfg=None):
    logger = logging.getLogger("pad")
    m = prepare_eval_model(model)
    rows = []
    for d in eval_domains:
        cfg = build_cfg(d, False)
        if extra_cfg:
            for k, v in extra_cfg.items():
                cfg.EVAL[k] = v
        cfg.DATASETS.NAMES = (d,)
        ds = build_dataset(d, cfg, verbose=False)
        _, _, val_loader, num_query, _ = make_dataloader(cfg, dataset=ds)
        t0 = time.time()
        r1, r5, mAP = do_inference(cfg, m, val_loader, num_query)
        logger.info(f"[Eval] trained={trained_domain} eval={d} Rank-1={r1:.4f} Rank-5={r5:.4f} mAP={mAP:.4f} "
                    f"({time.time() - t0:.0f}s)")
        row = {"trained_domain": trained_domain, "eval_domain": d, "rank1": f"{r1:.4f}", "rank5": f"{r5:.4f}",
               "map": f"{mAP:.4f}", "split_id": (extra_cfg or {}).get("SPLIT_ID", "")}
        rows.append(row)
        if csv_out:
            new = not osp.exists(csv_out)
            with open(csv_out, "a", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(row.keys()))
                if new:
                    w.writeheader()
                w.writerow(row)
    del m
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return rows

## ▶ Run the lifelong sequence
This is the long cell. Each finished domain leaves `PAD_outputs/<domain>/ViT-B-16_visual.pth`, and with `RESUME=True`
a rerun skips finished domains and their evaluations.

In [ ]:
SEQ_CSV = osp.join(OUT_DIR, "seq_eval.csv")


def _done_evals():
    if not osp.exists(SEQ_CSV):
        return set()
    with open(SEQ_CSV) as f:
        return {(r["trained_domain"], r["eval_domain"]) for r in csv.DictReader(f)}


prev_ckpt, t_all = None, time.time()
for idx, domain in enumerate(TRAIN_ORDER):
    if domain not in PREPARED:
        print(f"!! stopping: dataset '{domain}' is not available (see the data cell).")
        break
    existing = find_existing_ckpt(domain) if RESUME else None
    model = None
    if existing:
        print(f"[skip] {domain}: found checkpoint {existing}")
    else:
        t0 = time.time()
        model, _ = train_one_domain(domain, prev_ckpt, is_first=(idx == 0))
        print(f"[time] {domain} trained in {timedelta(seconds=int(time.time() - t0))}")
        existing = ckpt_path(domain)
    seen = TRAIN_ORDER[: idx + 1]
    todo = [d for d in seen if (domain, d) not in _done_evals() and d in PREPARED]
    if todo:
        if model is None:
            model = model_from_ckpt(existing, domain)
        setup_logger(osp.join(OUT_DIR, domain), "test_log.txt")
        evaluate_domains(model, todo, domain, csv_out=SEQ_CSV)
    prev_ckpt = existing
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print(f"[DONE] total wall time {timedelta(seconds=int(time.time() - t_all))}; results -> {SEQ_CSV}")

# Extra final evaluation when the prompt pool is not full (e.g. kaggle-4 fills 8+8+8+4 = 28 of 36 slots):
# the official test script counts all 36 slots, so 8 never-trained random slots also take part in top-k selection.
# Here the final model is ALSO evaluated with only the slots it actually trained, and both results are reported.
SEQ_CSV_TRAINSLOTS = osp.join(OUT_DIR, "final_eval_trained_slots.csv")
_done_domains = [d for d in TRAIN_ORDER if find_existing_ckpt(d)]
if _done_domains and _done_domains[-1] == TRAIN_ORDER[-1] and not osp.exists(SEQ_CSV_TRAINSLOTS):
    _fm = model_from_ckpt(ckpt_path(TRAIN_ORDER[-1]), TRAIN_ORDER[-1])
    _va = getattr(_fm.image_encoder, "va_prompt", None)
    if _va is not None and int(_va.active_sizes[0, 0]) < _va.pool_size:
        print(f"[extra eval] pool used {int(_va.active_sizes[0, 0])}/{_va.pool_size} slots -> evaluating with trained slots only")
        _flag = OFFICIAL_EVAL_SLOT_RECONSTRUCTION
        OFFICIAL_EVAL_SLOT_RECONSTRUCTION = False
        try:
            setup_logger(osp.join(OUT_DIR, TRAIN_ORDER[-1]), "test_log.txt")
            evaluate_domains(_fm, [d for d in TRAIN_ORDER if d in PREPARED], TRAIN_ORDER[-1], csv_out=SEQ_CSV_TRAINSLOTS)
        finally:
            OFFICIAL_EVAL_SLOT_RECONSTRUCTION = _flag
    del _fm

## Unseen-domain generalisation (port of `scripts/eval_unseen.sh`)
Protocol per dataset, matching Table 1's Unseen-Avg: CUHK02 and SenseReID use 1 deterministic split, GRID / i-LIDS /
PRID average 10 splits, and VIPeR / CUHK01 average 20 runs (10 identity splits × 2 directions). Seed 123.
Unseen-Avg is the mean over the datasets that are available.

In [ ]:
UNSEEN_PROTOCOL = {"cuhk02": 1, "sensereid": 1, "grid": 10, "ilids": 10, "prid": 10, "viper": 20, "cuhk01": 20}
UNSEEN_CSV = osp.join(OUT_DIR, "unseen_raw.csv")
unseen_summary = {}
last_domain = [d for d in TRAIN_ORDER if find_existing_ckpt(d)]
avail = [d for d in UNSEEN_PROTOCOL if d in PREPARED]
if RUN_UNSEEN_EVAL and last_domain and avail:
    last_domain = last_domain[-1]
    um = model_from_ckpt(ckpt_path(last_domain), last_domain)
    setup_logger(OUT_DIR, "unseen_log.txt")
    if osp.exists(UNSEEN_CSV):
        os.remove(UNSEEN_CSV)
    for d in avail:
        n = UNSEEN_PROTOCOL[d] if RUN_MODE == "full" else 1
        rows = []
        for sid in range(n):
            rows += evaluate_domains(um, [d], last_domain, csv_out=UNSEEN_CSV,
                                     extra_cfg={"SPLIT_ID": sid, "DIRECTION": "auto", "SEED": 123})
        unseen_summary[d] = (100 * statistics.mean(float(r["rank1"]) for r in rows),
                             100 * statistics.mean(float(r["map"]) for r in rows))
    del um
    for d, (r1, mp) in unseen_summary.items():
        print(f"{d:10s} Rank-1 {r1:6.2f}  mAP {mp:6.2f}")
    print(f"Unseen-Avg ({len(unseen_summary)} sets): Rank-1 {statistics.mean(v[0] for v in unseen_summary.values()):.2f}  "
          f"mAP {statistics.mean(v[1] for v in unseen_summary.values()):.2f}")
    if len(unseen_summary) < 7:
        print(f"NOTE: averaged over {len(unseen_summary)} of the paper's 7 unseen sets "
              f"({', '.join(sorted(unseen_summary))}); compare per dataset, not with the paper's 7-set Unseen-Avg.")
    with open(osp.join(OUT_DIR, "unseen_summary.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["dataset", "runs", "rank1", "map"])
        for d, (r1, mp) in unseen_summary.items():
            w.writerow([d, UNSEEN_PROTOCOL[d] if RUN_MODE == "full" else 1, f"{r1:.2f}", f"{mp:.2f}"])
else:
    print("unseen evaluation skipped (no unseen datasets attached, or no trained checkpoint yet)")

## Results: forgetting matrix, final table, and comparison with the paper
* **Matrix**: row = after training step *t*, column = evaluation domain. Values on the diagonal are just-learned
  performance. Reading down a column shows how much of that domain is **forgotten** later.
* **Final row** (after CUHK03): the numbers the paper reports in Table 1, i.e. per-domain mAP/R1 and Seen-Avg.
  With `PROTOCOL="kaggle-4"`, the comparison uses the paper's numbers for the same 4 domains. Those numbers come
  from the 5-domain run, so the comparison is close but not exact (see the note printed below the table).

`PAPER_REPORTED` holds the PAD numbers for AKA order-1 as quoted from the paper (arXiv 2605.05027). **Check them
against Table 1 of the PDF before you report anything.** The paper also gives a 7-seed mean of 70.30 ± 0.49 mAP /
80.98 ± 0.31 R1 for Seen-Avg, so a single-seed reproduction within about ±1 point is in the expected range.

In [ ]:
import matplotlib.pyplot as plt

PAPER_REPORTED = {  # (mAP, Rank-1) in %, PAD, AKA order-1, final step
    "market1501": (77.5, 89.8), "cuhksysu": (93.9, 94.9), "dukemtmcreid": (71.1, 82.6),
    "msmt17": (37.7, 64.0), "cuhk03": (66.4, 68.5),
}
PAPER_SEEN_AVG_7SEEDS = (70.30, 80.98)
SHORT = {"market1501": "Market", "cuhksysu": "CUHK-SYSU", "dukemtmcreid": "Duke", "msmt17": "MSMT17", "cuhk03": "CUHK03"}

res = defaultdict(dict)
if osp.exists(SEQ_CSV):
    with open(SEQ_CSV) as f:
        for r in csv.DictReader(f):
            res[r["trained_domain"]][r["eval_domain"]] = (100 * float(r["map"]), 100 * float(r["rank1"]))
steps = [d for d in TRAIN_ORDER if d in res]

if steps:
    hdr = "after step".ljust(12) + "".join(f"{SHORT[d]:>18s}" for d in TRAIN_ORDER) + f"{'Seen-Avg':>18s}"
    print("mAP / Rank-1 (%)\n" + hdr)
    for s in steps:
        vals = [res[s].get(d) for d in TRAIN_ORDER]
        got = [v for v in vals if v]
        line = SHORT[s].ljust(12) + "".join(f"{v[0]:8.1f} / {v[1]:5.1f} " if v else f"{'-':>18s}" for v in vals)
        line += f"{np.mean([v[0] for v in got]):8.1f} / {np.mean([v[1] for v in got]):5.1f}"
        print(line)

    final = steps[-1]
    print(f"\nFinal step ({SHORT[final]}) vs paper:")
    print(f"{'domain':12s}{'ours mAP':>10s}{'paper':>8s}{'Δ':>7s}{'ours R1':>10s}{'paper':>8s}{'Δ':>7s}")
    ours_m, ours_r = [], []
    for d in TRAIN_ORDER:
        if d in res[final]:
            m, r = res[final][d]; pm, pr = PAPER_REPORTED[d]; ours_m.append(m); ours_r.append(r)
            print(f"{SHORT[d]:12s}{m:10.1f}{pm:8.1f}{m - pm:+7.1f}{r:10.1f}{pr:8.1f}{r - pr:+7.1f}")
    if ours_m:
        doms_f = [d for d in TRAIN_ORDER if d in res[final]]
        pm = np.mean([PAPER_REPORTED[d][0] for d in doms_f]); pr = np.mean([PAPER_REPORTED[d][1] for d in doms_f])
        print(f"{'Seen-Avg':12s}{np.mean(ours_m):10.1f}{pm:8.1f}{np.mean(ours_m) - pm:+7.1f}"
              f"{np.mean(ours_r):10.1f}{pr:8.1f}{np.mean(ours_r) - pr:+7.1f}   (average over the {len(doms_f)} domains above)")
        if len(doms_f) == 5:
            print(f"paper 7-seed Seen-Avg: {PAPER_SEEN_AVG_7SEEDS}")
        else:
            print("NOTE: the paper's numbers come from the full 5-domain sequence. With a domain left out there is one\n"
                  "fewer step to forget through, so domains trained before the gap (e.g. Market, CUHK-SYSU, Duke) may score\n"
                  "somewhat HIGHER than the paper. Treat this as a close check, not an exact match.")

    tsc = osp.join(OUT_DIR, "final_eval_trained_slots.csv")
    if osp.exists(tsc):
        with open(tsc) as f:
            alt = {r["eval_domain"]: (100 * float(r["map"]), 100 * float(r["rank1"])) for r in csv.DictReader(f)}
        print(f"\nFinal model, evaluated with ONLY its trained prompt slots (pool not full; official script uses all 36):")
        for d in TRAIN_ORDER:
            if d in alt and d in res[final]:
                print(f"  {SHORT[d]:10s} mAP {alt[d][0]:5.1f} (official eval {res[final][d][0]:5.1f})   "
                      f"R1 {alt[d][1]:5.1f} (official eval {res[final][d][1]:5.1f})")
        print(f"  {'Seen-Avg':10s} mAP {np.mean([v[0] for v in alt.values()]):5.1f}   R1 {np.mean([v[1] for v in alt.values()]):5.1f}")

    # Forgetting: best-ever mAP on a domain minus its final mAP (only domains seen before the last step).
    print("\nForgetting (best mAP during the sequence − final mAP):")
    for d in TRAIN_ORDER[:-1]:
        hist = [res[s][d][0] for s in steps if d in res[s]]
        if len(hist) > 1:
            print(f"  {SHORT[d]:10s} {max(hist) - hist[-1]:5.1f} pts  (history: {', '.join(f'{h:.1f}' for h in hist)})")

    # ---- Figure 1: mAP matrix heatmap (sequential single hue) ----
    M = np.full((len(steps), len(TRAIN_ORDER)), np.nan)
    for i, s in enumerate(steps):
        for j, d in enumerate(TRAIN_ORDER):
            if d in res[s]:
                M[i, j] = res[s][d][0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})
    ax = axes[0]
    im = ax.imshow(M, cmap="Blues", vmin=0, vmax=100)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if not np.isnan(M[i, j]):
                ax.text(j, i, f"{M[i, j]:.1f}", ha="center", va="center", fontsize=9,
                        color="white" if M[i, j] > 60 else "#0b0b0b")
    ax.set_xticks(range(len(TRAIN_ORDER)), [SHORT[d] for d in TRAIN_ORDER], rotation=20)
    ax.set_yticks(range(len(steps)), [f"after {SHORT[s]}" for s in steps])
    ax.set_title("mAP (%) after each lifelong step", loc="left", fontsize=11)
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)

    # ---- Figure 2: final per-domain mAP, this run vs paper (grouped bars) ----
    ax = axes[1]
    doms = [d for d in TRAIN_ORDER if d in res[final]]
    x = np.arange(len(doms)); w = 0.38
    ours = [res[final][d][0] for d in doms]; paper = [PAPER_REPORTED[d][0] for d in doms]
    b1 = ax.bar(x - w / 2 - 0.01, ours, w, color="#2a78d6", label="This run")
    b2 = ax.bar(x + w / 2 + 0.01, paper, w, color="#eb6834", label="Paper (reported)")
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1, f"{b.get_height():.1f}",
                    ha="center", va="bottom", fontsize=8, color="#52514e")
    ax.set_xticks(x, [SHORT[d] for d in doms])
    ax.set_ylabel("mAP (%)"); ax.set_ylim(0, 105)
    ax.set_title(f"Final mAP after {SHORT[final]} ({PROTOCOL}): reproduction vs paper (5-domain)", loc="left", fontsize=11)
    ax.grid(axis="y", color="#e6e5e0", linewidth=0.8); ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.legend(frameon=False, loc="upper right")
    plt.tight_layout()
    fig.savefig(osp.join(OUT_DIR, "results.png"), dpi=150)
    plt.show()
else:
    print("no results yet")

if RUN_MODE == "quick":
    print("\n(quick mode: tiny schedule + subsampled evaluation, so these numbers are NOT comparable to the paper)")

## Outputs
Everything lives in `/kaggle/working/PAD_outputs/` (kept after a committed run):
`seq_eval.csv` (all seen-domain evals), `unseen_raw.csv`, `results.png`, `<domain>/train_log.txt`, and
`<domain>/ViT-B-16_visual.pth` (about 350 MB each; needed to resume or to re-run the unseen evaluation).